# 12 Tables and figures

This notebook produces every table and figure of the manuscript on walking access to food in the 92 metropolitan areas of Mexico. It reads the results files written by notebooks 05 to 11 and 13 and writes LaTeX table fragments, 300 dpi figures and supplementary CSV files directly to the project folder, so that no number in the manuscript is transcribed by hand.

Every estimate uses the estimation sample marked in the tract file: the tracts carrying the marginalisation index, a positive population and density, and a distance under every one of the thirteen definitions. The index enters on the scale standardised over that sample with population weights.

The notebook adds estimations of its own. Moran's I measures how strongly distance and the residuals of the gradient model cluster within metropolitan areas, which is what motivates clustering the standard errors on areas. The vehicle availability table reads distance together with the share of dwellings without a car, and relates the deprivation gradient to that share within areas. Where notebook 13 has been run, the robustness table also carries the gradient refitted on walking times adjusted for slope. The median regression estimates the distance of the median resident directly.

Outputs, under `Results/paper/`:

| folder | content |
| --- | --- |
| `tables_main/` | `t1_definitions.tex`, `t2_access.tex`, `t3_gradient.tex`, `t4_ranks.tex` and `t6_vehicle.tex`, one `table` environment each, ready for `\input{}` |
| `tables_appendix/` | `a1_robustness.tex`, `a2_strata.tex` |
| `supplement/` | `s1_area_level.csv`, `s2_per_area_ranks.csv`, `s3_coverage.csv`, `s4_value_registry.csv`, `s6_attrition.csv` |
| `figures/` | `f2_gradient.png`, `f3_maps.png`, `f4_scatter.png`, and the four workflow images `img_blocks.png`, `img_outlets.png`, `img_network.png`, `img_distance.png` |

Every quantity placed in a table is also written to `s4_value_registry.csv` with the file and column it came from. The run writes one consolidated log, `run_log_12_tables_figures.md`, and a zip of the `paper` folder, both offered for download at the end.

## 1. Setup

Paths, settings and the definitions of the thirteen destination sets. The order of `SETS` is the order used in every table and figure: each set is followed by its street commerce variant where one exists.

In [ ]:
import os, re, io, json, math, time, shutil, zipfile, warnings
import numpy as np
import pandas as pd
from scipy import stats, sparse
from scipy.spatial import cKDTree
from scipy.optimize import linprog
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.transforms

warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive")

BASE = os.environ.get("MXO_BASE", "/content/drive/MyDrive/Mexico_Obesity")
RESULTS = os.path.join(BASE, "Results")
PAPER = os.path.join(RESULTS, "paper")
DIRS = {k: os.path.join(PAPER, k) for k in ("tables_main", "tables_appendix", "supplement", "figures", "cache")}
def visible(path, tries=6):
    """Google Drive can be slow to show files in a freshly mounted folder; listing the folder refreshes it."""
    for k in range(tries):
        if os.path.exists(path):
            return True
        parent = os.path.dirname(path)
        try:
            os.listdir(parent)
        except OSError:
            pass
        time.sleep(2)
    return os.path.exists(path)

if not visible(RESULTS):
    if IN_COLAB:
        drive.mount("/content/drive", force_remount=True)
    if not visible(RESULTS):
        here = os.path.dirname(RESULTS)
        raise FileNotFoundError(f"{RESULTS} is not visible. Contents of {here}: "
                                + (", ".join(sorted(os.listdir(here))) if os.path.isdir(here) else "folder not found"))
print(f"Results folder: {RESULTS} ({len(os.listdir(RESULTS))} entries)")
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
OUT_LOCAL = "/content" if IN_COLAB else PAPER
REPORT_PATH = os.path.join(OUT_LOCAL, "run_log_12_tables_figures.md")
ZIP_PATH = os.path.join(OUT_LOCAL, "12_paper_outputs.zip")

SEED = 20261108
N_PERM = 999          # permutation draws for Moran's I and the within-area rank correlations
N_PERM_CITY = 9999    # permutation draws for the rank correlation across the 92 areas
N_BOOT = 499          # bootstrap resamples of whole areas for the median regression
NEIGHBOURS = 8        # nearest tracts within the same area
KERNEL_KM = (1, 2, 5) # Bartlett kernel cutoffs for spatially corrected errors
MILE_M = 1609.344
SHORT_M = 500.0
COMPLETE_MIN = 0.99

SETS = [
    "supermarket_only", "supermarket_club", "supermarket_mini", "supermarket_mini_pharmacy",
    "fresh_perishable", "fresh_perishable_street", "fresh_only", "fresh_street", "fresh_counter",
    "fresh_abarrotes", "fresh_abarrotes_street", "all_food", "all_food_street",
]
STREET_BASE = {
    "fresh_perishable_street": "fresh_perishable",
    "fresh_street": "fresh_only",
    "fresh_abarrotes_street": "fresh_abarrotes",
    "all_food_street": "all_food",
}
LABEL = {
    "supermarket_only": "Supermarkets",
    "supermarket_club": "Supermarkets and clubs",
    "supermarket_mini": "Supermarkets, clubs and small format",
    "supermarket_mini_pharmacy": "Supermarkets, clubs, small format and pharmacy minimarkets",
    "fresh_perishable": "Perishable specialists",
    "fresh_perishable_street": "Perishable specialists and street commerce",
    "fresh_only": "Fresh and staple specialists",
    "fresh_street": "Fresh and staple specialists and street commerce",
    "fresh_counter": "Fresh and staple specialists and craft dairy",
    "fresh_abarrotes": "Fresh specialists and grocery stores",
    "fresh_abarrotes_street": "Fresh specialists, grocery stores and street commerce",
    "all_food": "All food retail and food service",
    "all_food_street": "All food retail, food service and street commerce",
}
SHORT = {
    "fresh_thinned": "Fresh thinned to supermarket count",
    "random_nodes": "Random network nodes",
    "supermarket_only": "Supermarkets",
    "supermarket_club": "Supermarkets and clubs",
    "supermarket_mini": "+ small format",
    "supermarket_mini_pharmacy": "+ small format, pharmacy",
    "fresh_perishable": "Perishable specialists",
    "fresh_perishable_street": "Perishable + street",
    "fresh_only": "Fresh and staple specialists",
    "fresh_street": "Fresh + street",
    "fresh_counter": "Fresh + craft dairy",
    "fresh_abarrotes": "Fresh + grocery",
    "fresh_abarrotes_street": "Fresh + grocery + street",
    "all_food": "All food",
    "all_food_street": "All food + street",
}
CONTENTS = {
    "supermarket_only": "Full format self-service stores, warehouse clubs excluded",
    "supermarket_club": "Full format self-service stores and warehouse clubs",
    "supermarket_mini": "The above with small format self-service stores",
    "supermarket_mini_pharmacy": "The above with pharmacy minimarkets",
    "fresh_perishable": "Specialist sellers of perishables, tortillerías and bakeries excluded",
    "fresh_perishable_street": "Perishable specialists with street commerce blocks",
    "fresh_only": "Specialist fresh and staple sellers, tortillerías and traditional bakeries",
    "fresh_street": "Fresh and staple specialists with street commerce blocks",
    "fresh_counter": "Fresh and staple specialists with craft dairy producers",
    "fresh_abarrotes": "Fresh and staple specialists with neighbourhood grocery stores",
    "fresh_abarrotes_street": "The above with street commerce blocks",
    "all_food": "Every set above together with the remaining food retail and food service trades; bars and nightclubs excluded",
    "all_food_street": "The above with street commerce blocks",
}
BRANCH = {s: ("Self-service" if s.startswith("supermarket") else "Traditional" if s.startswith("fresh") else "Union")
          for s in SETS}
for s in STREET_BASE:
    BRANCH[s] = BRANCH[STREET_BASE[s]] + ", street"
ROB_PAIRS = {"supermarket_club_noshared": "supermarket_club", "fresh_only_noshared": "fresh_only"}
ANCHORS = ("supermarket_club", "fresh_only")
PAIRS = [
    ("supermarket_club", "fresh_perishable"),
    ("supermarket_club", "fresh_only"),
    ("supermarket_club", "fresh_street"),
    ("supermarket_club", "fresh_abarrotes"),
    ("supermarket_club", "all_food"),
    ("supermarket_mini", "fresh_perishable"),
    ("supermarket_mini", "fresh_only"),
    ("fresh_perishable", "fresh_only"),
]
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7, "xtick.labelsize": 6.5,
    "ytick.labelsize": 6.5, "legend.fontsize": 6.5, "axes.spines.top": False,
    "axes.spines.right": False, "savefig.dpi": 300, "figure.dpi": 110,
})
COL_W = 3.5  # single column width in inches

## 2. Report, registry and helpers

`Report` writes everything the notebook prints to one markdown file. `reg` records each number placed in a table or figure with its source, which becomes `s4_value_registry.csv`.

In [ ]:
class Report:
    def __init__(self, path):
        self.path, self.lines, self.t0 = path, [], time.time()
    def emit(self, s=""):
        print(s)
        self.lines.append(s)
    def h(self, text, level=2):
        self.emit("")
        self.emit("#" * level + " " + text)
        self.emit("")
    def p(self, text=""):
        self.emit(text)
    def df(self, d, digits=4, max_rows=300):
        d = d.head(max_rows)
        def f(v):
            if isinstance(v, (float, np.floating)):
                if not np.isfinite(v):
                    return ""
                return f"{v:,.{digits}f}" if abs(v) < 1e6 else f"{v:,.0f}"
            if isinstance(v, (int, np.integer)):
                return f"{v:,}"
            return str(v)
        cols = [str(c) for c in d.columns]
        self.emit("| " + " | ".join(cols) + " |")
        self.emit("| " + " | ".join("---" for _ in cols) + " |")
        for _, row in d.iterrows():
            self.emit("| " + " | ".join(f(v) for v in row.values) + " |")
        self.emit("")
    def save(self):
        self.emit("")
        self.emit(f"Run time {time.time() - self.t0:,.0f} s.")
        with open(self.path, "w", encoding="utf-8") as fh:
            fh.write("\n".join(self.lines) + "\n")

R = Report(REPORT_PATH)
R.h("Notebook 12, tables and figures", 1)
R.p(f"Base folder `{BASE}`. Seed {SEED}, {N_PERM} permutations ({N_PERM_CITY} across areas), {N_BOOT} bootstrap resamples, {NEIGHBOURS} neighbours, kernel cutoffs {KERNEL_KM} km.")

REG = []
def reg(key, value, source, column, description):
    try:
        v = float(value)
    except (TypeError, ValueError):
        v = np.nan
    REG.append(dict(key=key, value=v, source_file=source, source_column=column, description=description))
    return value

COLMAP = {}
def find_col(df, patterns, what, src, exclude=None, required=True):
    cols = [str(c) for c in df.columns]
    for pat in patterns:
        for c in cols:
            if re.search(pat, c, flags=re.I) and not (exclude and re.search(exclude, c, flags=re.I)):
                COLMAP[f"{src} :: {what}"] = c
                return c
    COLMAP[f"{src} :: {what}"] = None
    if required:
        raise KeyError(f"{src}: no column found for {what}; columns are {cols}")
    return None

def set_columns(df):
    out = []
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype):
            v = s.dropna().astype(str)
            if len(v) and v.isin(SETS).mean() >= 0.8:
                out.append(c)
    return out

def nrows_parquet(path):
    try:
        import pyarrow.parquet as pq
        return pq.ParquetFile(path).metadata.num_rows
    except Exception:
        return len(pd.read_parquet(path))

INPUTS = {}
def load(name, columns=None, required=True):
    path = os.path.join(RESULTS, name)
    if not visible(path, tries=3 if required else 1):
        if required:
            raise FileNotFoundError(path)
        R.p(f"- `{name}` not found")
        return None
    df = pd.read_parquet(path, columns=columns)
    INPUTS[name] = df
    R.p(f"- `{name}`: {len(df):,} rows; columns {', '.join(map(str, df.columns))}")
    return df

def wmedian(x, w):
    x = np.asarray(x, float); w = np.asarray(w, float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not m.any():
        return np.nan
    x, w = x[m], w[m]
    o = np.argsort(x, kind="stable")
    cw = np.cumsum(w[o])
    return float(x[o][np.searchsorted(cw, 0.5 * cw[-1])])

def wcorr(a, b, w):
    m = np.isfinite(a) & np.isfinite(b) & np.isfinite(w)
    a, b, w = a[m], b[m], w[m]
    ma, mb = np.average(a, weights=w), np.average(b, weights=w)
    return float(np.sum(w * (a - ma) * (b - mb)) / np.sqrt(np.sum(w * (a - ma) ** 2) * np.sum(w * (b - mb) ** 2)))

def holm(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    adj = np.empty(n); run = 0.0
    for i, k in enumerate(o):
        run = max(run, (n - i) * p[k]); adj[k] = min(1.0, run)
    return adj

def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)[::-1]
    adj = np.empty(n); run = 1.0
    for i, k in enumerate(o):
        run = min(run, p[k] * n / (n - i)); adj[k] = run
    return adj

def pnorm2(t):
    return 2.0 * stats.norm.sf(np.abs(t))

### Estimation helpers

`fit` is weighted least squares with standard errors clustered on metropolitan areas. With `fe=True` the area fixed effects are absorbed by subtracting the population weighted mean of each area from every variable, which gives the same coefficients as including one indicator per area. The small sample factor is G/(G−1)·(N−1)/(N−K). Probabilities use the normal distribution. `cross_cov` returns the clustered covariance between coefficients of two equations fitted on the same tracts, used for the difference between two definitions.

In [ ]:
def demean(A, w, g):
    A = np.asarray(A, float)
    if A.ndim == 1:
        A = A[:, None]
    sw = np.bincount(g, weights=w)
    out = np.empty_like(A)
    for j in range(A.shape[1]):
        m = np.bincount(g, weights=w * A[:, j]) / sw
        out[:, j] = A[:, j] - m[g]
    return out

def fit(y, X, w, g, fe=True):
    y = np.asarray(y, float); X = np.asarray(X, float)
    if X.ndim == 1:
        X = X[:, None]
    g = pd.factorize(np.asarray(g))[0]
    if fe:
        yd, Xd = demean(y, w, g)[:, 0], demean(X, w, g)
    else:
        yd, Xd = y, np.column_stack([X, np.ones(len(y))])
    sw = np.sqrt(w)
    beta = np.linalg.lstsq(Xd * sw[:, None], yd * sw, rcond=None)[0]
    e = yd - Xd @ beta
    bread = np.linalg.inv((Xd * w[:, None]).T @ Xd)
    S = Xd * (w * e)[:, None]
    G = int(g.max()) + 1
    Sg = np.zeros((G, Xd.shape[1])); np.add.at(Sg, g, S)
    n, k = Xd.shape
    c = G / (G - 1) * (n - 1) / (n - k)
    V = c * bread @ (Sg.T @ Sg) @ bread
    se = np.sqrt(np.diag(V))
    return dict(beta=beta, se=se, t=beta / se, p=pnorm2(beta / se), V=V, resid=e, Xd=Xd,
                bread=bread, Sg=Sg, c=c, n=n, G=G, w=w, g=g)

def cross_cov(f1, f2, j1=0, j2=0):
    C = f1["c"] * f1["bread"] @ (f1["Sg"].T @ f2["Sg"]) @ f2["bread"]
    return float(C[j1, j2])

def centre_within(v, g):
    m = np.bincount(g, weights=v) / np.bincount(g)
    return v - m[g]

def moran(z, W):
    return float(z @ (W @ z) / (z @ z))

def permute_within(n, g, order):
    perm = np.lexsort((rng.random(n), g))
    out = np.empty(n, dtype=int)
    out[order] = perm
    return out

def moran_test(v, W, g, n_perm=N_PERM):
    z = centre_within(np.asarray(v, float), g)
    I = moran(z, W)
    order = np.argsort(g, kind="stable")
    sims = np.empty(n_perm)
    for r in range(n_perm):
        sims[r] = moran(z[permute_within(len(z), g, order)], W)
    return I, (1 + np.sum(sims >= I)) / (n_perm + 1), float(sims.mean())

def kernel_se(f, coords, cutoff_m):
    S = f["Xd"] * (f["w"] * f["resid"])[:, None]
    meat = S.T @ S
    g = f["g"]
    for a in np.unique(g):
        idx = np.flatnonzero(g == a)
        if len(idx) < 2:
            continue
        tree = cKDTree(coords[idx])
        pr = tree.query_pairs(cutoff_m, output_type="ndarray")
        if len(pr) == 0:
            continue
        i, j = idx[pr[:, 0]], idx[pr[:, 1]]
        d = np.sqrt(((coords[i] - coords[j]) ** 2).sum(1))
        k = (1.0 - d / cutoff_m)[:, None]
        M = (S[i] * k).T @ S[j]
        meat += M + M.T
    n, kk = f["Xd"].shape
    V = n / (n - kk) * f["bread"] @ meat @ f["bread"]
    return np.sqrt(np.diag(V))

def lad_fe(y, X, w, g):
    g = pd.factorize(g)[0]
    n = len(y); G = g.max() + 1
    D = sparse.csr_matrix((np.ones(n), (np.arange(n), g)), shape=(n, G))
    Xs = sparse.hstack([sparse.csr_matrix(X), D]).tocsr(); p = Xs.shape[1]
    I = sparse.identity(n, format="csr")
    A = sparse.hstack([Xs, -Xs, I, -I]).tocsc()
    c = np.r_[np.zeros(2 * p), w, w]
    r = linprog(c, A_eq=A, b_eq=y, bounds=(0, None), method="highs-ipm")
    if r.status != 0:
        return np.full(X.shape[1], np.nan)
    b = r.x[:p] - r.x[p:2 * p]
    return b[:X.shape[1]]

### LaTeX helpers

Each table is written as a single `table` environment holding a caption, a label, an `adjustbox` and a pipe delimited `tabular`, followed by notes in a footnote sized paragraph. No package or preamble lines are written.

In [ ]:
def tex(s):
    s = str(s)
    for a, b in (("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#")):
        s = s.replace(a, b)
    return s

def num(x, d=0):
    if x is None or not np.isfinite(x):
        return "--"
    s = f"{abs(x):,.{d}f}"
    neg = x < 0 and float(s.replace(",", "")) != 0
    return ("$-$" if neg else "") + s

def pct(x, d=1):
    return "--" if x is None or not np.isfinite(x) else f"{100 * x:.{d}f}"

def pfmt(p):
    if p is None or not np.isfinite(p):
        return "--"
    return "$<$0.001" if p < 0.001 else f"{p:.3f}"

def write_table(path, caption, label, colspec, header, body, notes=None):
    lines = [r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}", r"\label{" + label + "}",
             r"\begin{adjustbox}{max width=\textwidth}", r"\begin{tabular}{" + colspec + "}", r"\hline"]
    for h in header:
        lines.append(" & ".join(h) + r" \\")
    lines.append(r"\hline")
    for row in body:
        if isinstance(row, str):
            lines.append(row)
        else:
            lines.append(" & ".join(row) + r" \\")
    lines += [r"\hline", r"\end{tabular}", r"\end{adjustbox}"]
    if notes:
        lines += [r"\par\smallskip", r"\begin{minipage}{\textwidth}\footnotesize", notes, r"\end{minipage}"]
    lines.append(r"\end{table}")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")
    R.p(f"Wrote `{os.path.relpath(path, PAPER)}`.")

def panel_row(text, ncol):
    return r"\multicolumn{" + str(ncol) + r"}{|l|}{" + text + r"} \\ \hline"

## 3. Inputs

All inputs are results files of the pipeline. The report lists the rows and columns of each.

In [ ]:
R.h("Inputs")
tr = load("08_tract_access.parquet")
area = load("08_area_access.parquet")
blk = load("07_block_access.parquet")
rob = load("07_block_access_robustness.parquet")
att = load("07_network_attachment.parquet")
out = load("05_outlets.parquet")
stc = load("06_street_commerce.parquet")
t04 = load("04_tracts.parquet")
inf = load("11_inference.parquet")
con = load("11_contrasts.parquet")
spa = load("11_spatial.parquet")
gw = load("10_gradient_within.parquet", required=False)
dnull = load("10_density_null.parquet", required=False)

blk["CVEGEO"] = blk["CVEGEO_MZA"].astype(str).str[:13]
if not {"x_m", "y_m"} <= set(tr.columns):
    _b = blk.loc[blk["POBTOT"] > 0, ["CVEGEO", "POBTOT", "x_m", "y_m"]]
    _c = (_b.assign(wx=_b["POBTOT"] * _b["x_m"], wy=_b["POBTOT"] * _b["y_m"])
            .groupby("CVEGEO")[["wx", "wy", "POBTOT"]].sum())
    _c = pd.DataFrame({"x_m": _c["wx"] / _c["POBTOT"], "y_m": _c["wy"] / _c["POBTOT"]}).reset_index()
    tr = tr.merge(_c, on="CVEGEO", how="left")
    R.p(f"Tract centroids taken as the population weighted mean of block coordinates; {int(tr['x_m'].isna().sum())} tracts without one.")

def need(df, cols, name):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"{name} lacks {miss}")
need(tr, ["CVEGEO", "slug", "population", "imu_depriv_z", "pop_density_km2", "x_m", "y_m", "estimation_sample", "veh_share"]
     + [f"median_m_{s}" for s in SETS], "08_tract_access")
need(rob, ["CVEGEO_MZA", "POBTOT"] + [f"distance_m_{s}" for s in ROB_PAIRS], "07_block_access_robustness")
need(t04, ["CVEGEO", "veh_share"], "04_tracts")
need(blk, ["CVEGEO_MZA", "slug", "POBTOT"] + [f"distance_m_{s}" for s in SETS] + [f"walk_min_{s}" for s in SETS], "07_block_access")

slug_tipo = area.set_index("slug")["tipo"] if "tipo" in area.columns else pd.Series(dtype=object)
if "macro_region" in area.columns:
    slug_region = area.set_index("slug")["macro_region"]
elif "macro_region" in tr.columns:
    slug_region = tr.dropna(subset=["macro_region"]).groupby("slug")["macro_region"].agg(lambda v: v.mode().iloc[0])
else:
    slug_region = pd.Series(dtype=object)
name_col = find_col(area, [r"^name$", r"^nombre", r"nom_?met", r"metro_?name", r"area_?name", r"^nom_zm"], "area name", "08_area_access", required=False)
def pretty(slug):
    return str(slug).replace("_", " ").replace("-", " ").title()
AREA_NAME = (area.set_index("slug")[name_col] if name_col else pd.Series({s: pretty(s) for s in area["slug"]})).to_dict()

## 4. Estimation sample, neighbours and robustness medians

The sample is the one marked by the aggregation notebook, and the flag is checked against its own definition here. Tract centroids are the population weighted mean of block coordinates, neighbours for Moran's I are the eight nearest tract centroids within the same area, and the index is confirmed to have weighted mean zero and unit weighted spread on the sample.

The robustness distances written at block level, the anchor sets without establishments on shared coordinates, are carried up to the tract as population weighted medians over the same residents, so that they can be fitted with the same models as the main definitions.

In [ ]:
R.h("Estimation sample")
med_cols = [f"median_m_{s}" for s in SETS]
flag = tr["estimation_sample"].fillna(False).astype(bool)
crit = ((tr["population"] > 0) & tr["imu_depriv_z"].notna() & (tr["pop_density_km2"] > 0)
        & tr[med_cols].notna().all(axis=1))
if int((flag != crit).sum()):
    raise ValueError(f"the estimation_sample flag disagrees with its definition for {int((flag != crit).sum())} tracts")
ok = flag & tr["x_m"].notna() & tr["y_m"].notna()
R.p(f"{int(flag.sum()):,} tracts flagged, {int((flag & ~ok).sum())} of them without a centroid.")
d = tr.loc[ok].reset_index(drop=True)
gcode, AREAS = pd.factorize(d["slug"])
w = d["population"].to_numpy(float)
xdep = d["imu_depriv_z"].to_numpy(float)
logden = np.log(d["pop_density_km2"].to_numpy(float))
coords = d[["x_m", "y_m"]].to_numpy(float)
carless = 1.0 - d["veh_share"].to_numpy(float)
Y = {s: np.log1p(d[f"median_m_{s}"].to_numpy(float)) for s in SETS}
reg("inf.tracts", len(d), "08_tract_access.parquet", "estimation_sample", "tracts in the estimation sample")
reg("inf.population", w.sum(), "08_tract_access.parquet", "population", "population in the estimation sample")
reg("inf.areas", len(AREAS), "08_tract_access.parquet", "slug", "areas in the estimation sample")

wm = float(np.average(xdep, weights=w))
ws = float(math.sqrt(np.average((xdep - wm) ** 2, weights=w)))
R.p(f"Marginalisation index on the sample: weighted mean {wm:.2e}, weighted standard deviation {ws:.6f}.")
if abs(wm) > 1e-6 or abs(ws - 1) > 1e-6:
    raise ValueError("the index is not standardised over the estimation sample; run notebook 08 again")
R.p(f"Share of dwellings without a car is missing for {int(np.isnan(carless).sum())} tracts of the sample.")

rows, cols = [], []
for a in range(len(AREAS)):
    idx = np.flatnonzero(gcode == a)
    k = min(NEIGHBOURS, len(idx) - 1)
    if k < 1:
        continue
    _, nn = cKDTree(coords[idx]).query(coords[idx], k=k + 1)
    is_self = nn == np.arange(len(idx))[:, None]
    keep = ~is_self
    no_self = ~is_self.any(axis=1)
    keep[no_self, -1] = False
    nb = nn[keep].reshape(len(idx), k)
    rows.append(np.repeat(idx, k)); cols.append(idx[nb.ravel()])
rows, cols = np.concatenate(rows), np.concatenate(cols)
W = sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(d), len(d)))
W = sparse.diags(1.0 / np.asarray(W.sum(axis=1)).ravel()) @ W
reg("inf.links", W.nnz, "08_tract_access.parquet", "x_m, y_m", "neighbour links within areas")
R.p(f"{len(d):,} tracts, {w.sum():,.0f} people, {len(AREAS)} areas, {W.nnz:,} neighbour links.")

OWN = {s: fit(Y[s], np.column_stack([xdep, logden]), w, gcode) for s in SETS}

# robustness distances, block to tract
R.h("Robustness medians at tract level", 3)
rob["CVEGEO"] = rob["CVEGEO_MZA"].astype(str).str[:13]
ROB_COLS = [f"distance_m_{s}" for s in ROB_PAIRS]
def group_wmedian(frame, by, col, weight):
    f = frame[[by, col, weight]].dropna(subset=[col])
    f = f[f[weight] > 0].sort_values([by, col], kind="stable")
    tot = f.groupby(by, sort=False)[weight].transform("sum")
    cum = f.groupby(by, sort=False)[weight].cumsum()
    return f[cum >= 0.5 * tot].groupby(by, sort=False)[col].first()
RT = pd.DataFrame(index=d["CVEGEO"])
for c in ROB_COLS:
    RT[c] = group_wmedian(rob, "CVEGEO", c, "POBTOT").reindex(d["CVEGEO"]).to_numpy(float)
YR = {c[len("distance_m_"):]: np.log1p(RT[c].to_numpy(float)) for c in ROB_COLS}
miss = pd.Series({k: int(np.isnan(v).sum()) for k, v in YR.items()})
R.p("Tracts of the sample without a median under each robustness set:")
R.df(miss.rename("tracts").reset_index().rename(columns={"index": "set"}))
chk_rob = pd.DataFrame({"set": list(ROB_PAIRS), "median m, robustness": [wmedian(RT[f"distance_m_{s}"], w) for s in ROB_PAIRS],
                        "median m, main": [wmedian(d[f"median_m_{ROB_PAIRS[s]}"], w) for s in ROB_PAIRS]})
R.df(chk_rob)

def fit_on(y, X, mask=None):
    """The within-area model on the tracts where y is finite."""
    msk = np.isfinite(y) if mask is None else (mask & np.isfinite(y))
    f = fit(y[msk], X[msk], w[msk], gcode[msk])
    f["mask"] = msk
    return f

## 5. Table 3, the within-area gradient

The coefficients, clustered errors and probabilities are those written by the inference notebook. They are compared with the same model fitted here on the same sample, which confirms the specification. The adjusted probabilities of the inference notebook are read and summarised in the table note rather than tabulated, since every coefficient that is distinguishable from zero stays so under both corrections. The last column converts the coefficient to the per cent change in distance for one standard deviation of the index.

In [ ]:
R.h("Table 3, within-area gradient")
GR = None
try:
    sc = set_columns(inf)[0]
    excl_coef = r"(^se|_se|se_|^t_|^t$|^p_|^p$|pval|holm|bh|fdr|pooled|median|qr|retain|share|first|second)"
    c_b = find_col(inf, [r"^coef(_within)?$", r"^beta(_within)?$", r"^estimate$", r"^b_within$", r"within", r"coef", r"beta"], "coefficient", "11_inference", exclude=excl_coef)
    c_se = find_col(inf, [r"^standard error$", r"^se$", r"^se_?(within|clust|cluster|clustered)$", r"^se", r"std_?err", r"standard error"], "standard error", "11_inference",
                    exclude=r"(pooled|indep|unrel|naive|boot|km|median|qr)")
    c_t = find_col(inf, [r"^t$", r"^t_?(stat|within|value)$", r"^t_", r"tstat"], "t", "11_inference", exclude=r"(pooled|median)")
    c_p = find_col(inf, [r"^p$", r"^p_?(raw|value|val|within)$", r"^p_?(raw|value)", r"^pval"], "raw p", "11_inference",
                   exclude=r"(holm|bh|fdr|adj|pooled|median)")
    c_h = find_col(inf, [r"holm", r"any false finding", r"fwer"], "Holm p", "11_inference")
    c_q = find_col(inf, [r"(^|_)bh($|_)", r"fdr", r"benjamini", r"hochberg", r"share of false findings"], "BH p", "11_inference")
    I = inf.set_index(sc)
    GR = pd.DataFrame({"coef": I[c_b], "se": I[c_se], "t": I[c_t], "p": I[c_p], "p_holm": I[c_h], "p_bh": I[c_q]}).reindex(SETS)
    GR_SRC = ("11_inference.parquet", {"coef": c_b, "se": c_se, "t": c_t, "p": c_p, "p_holm": c_h, "p_bh": c_q})
    if GR.isna().any().any():
        raise ValueError("11_inference lacks rows for " + ", ".join(GR.index[GR.isna().any(axis=1)]))
except Exception as exc:
    R.p(f"The stored gradient could not be read ({exc}); Table 3 uses the fit made in this notebook.")
    GR = pd.DataFrame({s: dict(coef=OWN[s]["beta"][0], se=OWN[s]["se"][0], t=OWN[s]["t"][0], p=OWN[s]["p"][0]) for s in SETS}).T
    GR["p_holm"] = holm(GR["p"].to_numpy()); GR["p_bh"] = bh(GR["p"].to_numpy())
    GR_SRC = ("computed in notebook 12", {k: k for k in GR.columns})

chk = pd.DataFrame({"stored coef": GR["coef"], "refit coef": [OWN[s]["beta"][0] for s in SETS],
                    "stored se": GR["se"], "refit se": [OWN[s]["se"][0] for s in SETS],
                    "density coef": [OWN[s]["beta"][1] for s in SETS],
                    "Holm recomputed": holm(GR["p"].to_numpy()), "BH recomputed": bh(GR["p"].to_numpy())}, index=SETS)
chk.insert(0, "set", SETS)
R.p("Stored against refitted coefficients, and adjusted probabilities recomputed from the stored raw probabilities:")
R.df(chk.reset_index(drop=True))
max_gap = float(np.max(np.abs(chk["stored coef"] - chk["refit coef"])))
R.p(f"Largest gap between stored and refitted coefficients {max_gap:.4f}.")
if max_gap > 0.01:
    R.p("The refit differs from the stored estimates by more than 0.01, so the controls of this notebook do not match those of the inference notebook. Table 3 uses the stored estimates; Table 5 and the appendix use the refit and should be read with that difference in mind.")

for s in SETS:
    for k in ("coef", "se", "t", "p", "p_holm", "p_bh"):
        reg(f"t3.{k}.{s}", GR.loc[s, k], GR_SRC[0], GR_SRC[1][k], f"within-area gradient, {k}, {s}")

CT = None
try:
    sc2 = set_columns(con)
    if len(sc2) < 2:
        raise KeyError("two set name columns not found")
    a_col, b_col = sc2[0], sc2[1]
    x_se = r"(indep|unrel|naive|uncorr|_a$|_b$|_1$|_2$|narrow|wide|^se_?(a|b|1|2)$)"
    c_d = find_col(con, [r"^diff", r"difference", r"^delta"], "difference", "11_contrasts", exclude=r"(se|^t|^p|indep)")
    c_dse = find_col(con, [r"^standard error$", r"^se_?diff$", r"^se$", r"se_?diff", r"diff_?se", r"^se"], "difference se", "11_contrasts", exclude=x_se)
    c_dt = find_col(con, [r"^t$", r"^t_?diff", r"^t_?stat", r"^t_"], "difference t", "11_contrasts", exclude=r"(indep|unrel)")
    c_dp = find_col(con, [r"^p$", r"^p_?diff", r"^p_?value", r"^p_"], "difference p", "11_contrasts", exclude=r"(indep|unrel|holm|bh)")
    c_dc = find_col(con, [r"^corr", r"corr", r"^rho"], "correlation of estimates", "11_contrasts")
    rows_ = []
    for a, b in PAIRS:
        r_ = con[(con[a_col].astype(str) == a) & (con[b_col].astype(str) == b)]
        sign = 1.0
        if r_.empty:
            r_ = con[(con[a_col].astype(str) == b) & (con[b_col].astype(str) == a)]; sign = -1.0
        if r_.empty:
            raise KeyError(f"contrast {a} against {b} not in 11_contrasts")
        r_ = r_.iloc[0]
        rows_.append(dict(a=a, b=b, diff=sign * r_[c_d], se=r_[c_dse], t=sign * r_[c_dt], p=r_[c_dp], corr=r_[c_dc]))
    CT = pd.DataFrame(rows_)
    CT_SRC = ("11_contrasts.parquet", {"diff": c_d, "se": c_dse, "t": c_dt, "p": c_dp, "corr": c_dc})
except Exception as exc:
    R.p(f"The stored contrasts could not be read ({exc}); they are computed here from the joint clustered covariance.")
    rows_ = []
    for a, b in PAIRS:
        fa, fb = OWN[a], OWN[b]
        cov = cross_cov(fa, fb)
        dv = fa["beta"][0] - fb["beta"][0]
        se = math.sqrt(fa["V"][0, 0] + fb["V"][0, 0] - 2 * cov)
        rows_.append(dict(a=a, b=b, diff=dv, se=se, t=dv / se, p=pnorm2(dv / se), corr=cov / (fa["se"][0] * fb["se"][0])))
    CT = pd.DataFrame(rows_)
    CT_SRC = ("computed in notebook 12", {k: k for k in ("diff", "se", "t", "p", "corr")})
own_ct = []
for a, b in PAIRS:
    fa, fb = OWN[a], OWN[b]
    cov = cross_cov(fa, fb)
    own_ct.append(dict(diff_refit=fa["beta"][0] - fb["beta"][0], se_refit=math.sqrt(fa["V"][0, 0] + fb["V"][0, 0] - 2 * cov)))
R.df(pd.concat([CT, pd.DataFrame(own_ct)], axis=1))
for _, r_ in CT.iterrows():
    for k, kk in (("diff", "diff"), ("se", "diff_se"), ("t", "diff_t"), ("p", "diff_p"), ("corr", "diff_corr")):
        reg(f"t3.{kk}.{r_.a}.{r_.b}", r_[k], CT_SRC[0], CT_SRC[1][k], f"contrast {r_.a} minus {r_.b}, {k}")

for s in SETS:
    reg(f"t3.pct.{s}", 100 * (math.exp(GR.loc[s, "coef"]) - 1), GR_SRC[0], GR_SRC[1]["coef"], f"per cent change in distance per SD, {s}")
sig = GR["p"] < 0.05
surv = int(((GR["p_holm"] < 0.05) & (GR["p_bh"] < 0.05) & sig).sum())
R.p(f"{int(sig.sum())} of {len(SETS)} coefficients have p below 0.05, and {surv} of those stay below 0.05 under both the Holm and the "
    f"Benjamini and Hochberg corrections.")
reg("t3.significant", int(sig.sum()), GR_SRC[0], GR_SRC[1]["p"], "coefficients with p below 0.05")
reg("t3.survive", surv, GR_SRC[0], GR_SRC[1]["p_holm"], "of those, surviving both corrections")
not_sig = [LABEL[s] for s in SETS if not sig[s]]

head = [["Definition", r"Coefficient", r"Std. error", r"$t$", r"$p$", r"Change per SD (\%)"]]
body = [panel_row(r"Panel A. Coefficient on the marginalisation index, within areas", 6)]
for s in SETS:
    gg = GR.loc[s]
    body.append([tex(LABEL[s]), num(gg.coef, 4), num(gg.se, 4), num(gg.t, 2), pfmt(gg.p), num(100 * (math.exp(gg.coef) - 1), 1)])
body.append(r"\hline")
body.append(panel_row(r"Panel B. Difference between two definitions", 6))
body.append(r"Narrower minus wider & Difference & Std. error & $t$ & $p$ & Corr. of estimates \\ \hline")
for _, r_ in CT.iterrows():
    body.append([tex(f"{SHORT[r_.a]} minus {SHORT[r_.b]}"), num(r_["diff"], 4), num(r_["se"], 4), num(r_["t"], 2), pfmt(r_["p"]),
                 num(r_["corr"], 3)])
adj_note = (f"All {surv} coefficients with $p$ below 0.05 remain below 0.05 after the Holm and the Benjamini and Hochberg corrections "
            f"for the thirteen definitions" + (f"; the exception is {tex(not_sig[0])}, which is not distinguishable from zero before or after adjustment." if len(not_sig) == 1
            else (f"; the exceptions are {tex(', '.join(not_sig))}." if not_sig else ".")))
if surv != int(sig.sum()):
    adj_note = (f"{surv} of the {int(sig.sum())} coefficients with $p$ below 0.05 remain below 0.05 after the Holm and the Benjamini and "
                f"Hochberg corrections for the thirteen definitions.")
notes = (f"Weighted least squares of log(1 + tract median walking distance) on the marginalisation index, standardised over the sample "
         f"with population weights, and log population density, with metropolitan area fixed effects and weights equal to tract "
         f"population. Standard errors are clustered on the {len(AREAS)} areas. The last column is 100(exp(coefficient) $-$ 1), the per "
         f"cent change in the distance of the median resident for one standard deviation of the index. {adj_note} Panel B gives the "
         f"difference between the coefficients of two definitions, with its standard error taken from the joint clustered covariance "
         f"of the two estimates. Sample of {len(d):,} tracts holding {w.sum():,.0f} people.")
write_table(os.path.join(DIRS["tables_main"], "t3_gradient.tex"),
            "Within-area association between deprivation and walking distance to the nearest destination, by definition",
            "tab:gradient", "|l|r|r|r|r|r|", head, body, notes)

## 6. Spatial clustering of distance and residuals

Moran's I is computed for the two anchor definitions on the outcome and on the residuals of the within-area model of the gradient table, with each tract's neighbours the eight nearest tract centroids in the same area, values centred on their area mean and probabilities from 999 permutations of values within areas. Strong clustering of the residuals within areas is the reason inference throughout uses standard errors clustered on areas, and the values are given in the note of the robustness table beside the errors that allow correlation only within 1, 2 and 5 km. The permutations draw from a generator of their own, so the probabilities do not depend on what the notebook computed before.

In [ ]:
R.h("Spatial clustering")
rng = np.random.default_rng([SEED, 6])
MOR = {}
for s in ANCHORS:
    Iy, py, my = moran_test(Y[s], W, gcode)
    I0, p0, m0 = moran_test(OWN[s]["resid"], W, gcode)
    MOR[s] = dict(moran_y=Iy, p_y=py, null_mean_y=my, moran_base=I0, p_base=p0, null_mean_base=m0)
    reg(f"moran.y.{s}", Iy, "computed in notebook 12 from 08_tract_access.parquet", "median_m, x_m, y_m", f"Moran's I of the outcome, {s}")
    reg(f"moran.base.{s}", I0, "computed in notebook 12 from 08_tract_access.parquet", "median_m, imu_depriv_z, pop_density_km2, x_m, y_m",
        f"Moran's I of the baseline residuals, {s}")
    reg(f"moran.p_base.{s}", p0, "computed in notebook 12", f"{N_PERM} permutations within areas", f"permutation probability, baseline residuals, {s}")
R.df(pd.DataFrame(MOR).T.reset_index().rename(columns={"index": "set"}))

## 7. Table 4, rank agreement

At the city scale the 92 areas are ranked by their population weighted median distance under each definition, rank 1 being the longest walk, and the two orderings are compared with Spearman's correlation. The probability comes from 9,999 random rearrangements of one ordering. At the neighbourhood scale tracts of the estimation sample are ranked within their own area, Spearman's correlation is computed inside each area, and the mean over areas is tested by rearranging one ordering within every area. Sparse peripheral tracts are far from every kind of outlet, so two distance measures agree in part because both follow density; the partial correlation removes the rank of residential density from both orderings inside each area and is tested the same way. The pooled correlation over all tracts of the sample is shown between the two scales.

In [ ]:
R.h("Table 4, rank agreement")
rng = np.random.default_rng([SEED, 7])
amed = {}
for s in SETS:
    c = f"median_m_{s}" if f"median_m_{s}" in area.columns else find_col(area, [rf"median.*{s}$", rf"{s}.*median"], f"area median {s}", "08_area_access")
    amed[s] = area[c].to_numpy(float)
RANK = {s: pd.Series(amed[s], index=area["slug"]).rank(ascending=False, method="min") for s in SETS}

def group_pearson(u, v, g, G):
    n = np.bincount(g, minlength=G).astype(float)
    mu = np.bincount(g, u, G) / n; mv = np.bincount(g, v, G) / n
    du, dv = u - mu[g], v - mv[g]
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.bincount(g, du * dv, G) / np.sqrt(np.bincount(g, du * du, G) * np.bincount(g, dv * dv, G))

def group_partial(u, v, z, g, G):
    ruv, ruz, rvz = group_pearson(u, v, g, G), group_pearson(u, z, g, G), group_pearson(v, z, g, G)
    with np.errstate(invalid="ignore", divide="ignore"):
        return (ruv - ruz * rvz) / np.sqrt((1 - ruz ** 2) * (1 - rvz ** 2))

G = len(AREAS)
rank_den = pd.Series(logden).groupby(gcode).rank(pct=True).to_numpy()
order = np.argsort(gcode, kind="stable")
RK, PER_AREA = [], []
for a, b in PAIRS:
    xa, xb = amed[a], amed[b]
    mm = np.isfinite(xa) & np.isfinite(xb)
    rho_c = stats.spearmanr(xa[mm], xb[mm]).statistic
    ra, rb = stats.rankdata(xa[mm]), stats.rankdata(xb[mm])
    sims = np.array([np.corrcoef(ra, rng.permutation(rb))[0, 1] for _ in range(N_PERM_CITY)])
    p_c = (1 + np.sum(np.abs(sims) >= abs(rho_c))) / (N_PERM_CITY + 1)

    rho_t = stats.spearmanr(d[f"median_m_{a}"], d[f"median_m_{b}"]).statistic
    u = pd.Series(d[f"median_m_{a}"].to_numpy(float)).groupby(gcode).rank(pct=True).to_numpy()
    v = pd.Series(d[f"median_m_{b}"].to_numpy(float)).groupby(gcode).rank(pct=True).to_numpy()
    r_area = group_pearson(u, v, gcode, G)
    r_part = group_partial(u, v, rank_den, gcode, G)
    mean_w, mean_p = float(np.nanmean(r_area)), float(np.nanmean(r_part))
    sims_w, sims_p = np.empty(N_PERM), np.empty(N_PERM)
    for r in range(N_PERM):
        vp = v[permute_within(len(v), gcode, order)]
        sims_w[r] = np.nanmean(group_pearson(u, vp, gcode, G))
        sims_p[r] = np.nanmean(group_partial(u, vp, rank_den, gcode, G))
    p_w = (1 + np.sum(np.abs(sims_w - sims_w.mean()) >= abs(mean_w - sims_w.mean()))) / (N_PERM + 1)
    p_p = (1 + np.sum(np.abs(sims_p - sims_p.mean()) >= abs(mean_p - sims_p.mean()))) / (N_PERM + 1)
    RK.append(dict(a=a, b=b, city_rho=rho_c, city_p=p_c, areas=int(mm.sum()), tract_rho=rho_t, tracts=len(d),
                   within_rho=mean_w, within_median=float(np.nanmedian(r_area)), within_sd=float(np.nanstd(r_area, ddof=1)),
                   within_min=float(np.nanmin(r_area)), within_max=float(np.nanmax(r_area)), within_p=p_w,
                   within_neg=int(np.sum(r_area < 0)), within_partial=mean_p, within_partial_p=p_p,
                   within_partial_neg=int(np.sum(r_part < 0))))
    n_by = np.bincount(gcode, minlength=G)
    for i, sl in enumerate(AREAS):
        PER_AREA.append(dict(slug=sl, area=AREA_NAME.get(sl, pretty(sl)), set_a=a, set_b=b, tracts=int(n_by[i]), rho=r_area[i], rho_partial=r_part[i]))
RK = pd.DataFrame(RK)
R.df(RK)
for _, r_ in RK.iterrows():
    for k in ("city_rho", "city_p", "tract_rho", "within_rho", "within_p", "within_neg", "within_median", "within_sd",
              "within_partial", "within_partial_p", "within_partial_neg"):
        reg(f"t4.{k}.{r_.a}.{r_.b}", r_[k], "08_area_access.parquet / 08_tract_access.parquet", f"median_m_{r_.a}, median_m_{r_.b}", f"rank agreement {k}")
R.p(f"Within-area agreement between {SHORT[ANCHORS[0]]} and {SHORT[ANCHORS[1]]}: mean rho {RK.iloc[1].within_rho:.3f}, "
    f"shared rank variance about {100 * RK.iloc[1].within_rho ** 2:.0f} per cent; partial on density {RK.iloc[1].within_partial:.3f}.")
reg("t4.shared_variance.supermarket_club.fresh_only", 100 * RK.iloc[1].within_rho ** 2, "08_tract_access.parquet", "median_m", "squared mean within-area rho, per cent")

hd = [["", r"\multicolumn{2}{c|}{Between areas}", r"Pooled tracts", r"\multicolumn{5}{c|}{Within areas}"],
      ["Pair of definitions", r"$\rho$", r"$p$", r"$\rho$", r"Mean $\rho$", r"$p$", r"Mean partial $\rho$", r"$p$", r"Areas with $\rho<0$"]]
bd = [[tex(f"{SHORT[r_.a]} against {SHORT[r_.b]}"), num(r_.city_rho, 3), pfmt(r_.city_p), num(r_.tract_rho, 3),
       num(r_.within_rho, 3), pfmt(r_.within_p), num(r_.within_partial, 3), pfmt(r_.within_partial_p),
       f"{int(r_.within_neg)} of {int(RK.areas.max())}"] for _, r_ in RK.iterrows()]
notes = (f"Spearman correlations. Between areas, the {int(RK.areas.max())} metropolitan areas are ranked by their population weighted "
         f"median walking distance and the probability comes from {N_PERM_CITY:,} random rearrangements of one ordering. Pooled tracts "
         f"compares tract medians over the {len(d):,} tracts of the estimation sample. Within areas, tracts are ranked inside their own "
         f"area, the correlation is computed separately in each area and averaged, and the probability comes from {N_PERM} rearrangements "
         f"of one ordering within every area. The partial correlation removes the within-area rank of log population density from both "
         f"orderings before the correlation is taken, and is tested in the same way. All probabilities are two sided.")
write_table(os.path.join(DIRS["tables_main"], "t4_ranks.tex"),
            "Agreement between definitions on which places are furthest from food, between and within metropolitan areas",
            "tab:ranks", "|l|r|r|r|r|r|r|r|r|", hd, bd, notes)

## 8. Table 6, distance and vehicle availability

Walking distance constrains a household with no car and inconveniences one with a car, so the two are read together, in the form used by United States federal practice, where a tract is flagged on distance and the households without a vehicle beyond that distance are counted separately. Residents are divided into fifths of equal population by the share of dwellings in their tract without a car, and the distribution of distance across those fifths is shown. The share of residents beyond one mile from a supermarket and in a tract where most dwellings have no car is then compared, nationally and by macro region, with the share expected if the two conditions were unrelated. A final panel adds the carless share and its interaction with deprivation to the within-area model of the gradient table, for the two anchor definitions. Both variables are centred on their population weighted mean within each area before they are multiplied, so that the interaction compares tracts within the same city and does not pick up the difference between cities with many cars and cities with few. Because the carless share rises with deprivation, a positive interaction could also reflect a gradient that simply steepens at the deprived end of the index, so the model is refitted with the square of the index added and the report gives the interaction under both.

In [ ]:
R.h("Table 6, vehicle availability")
vb = blk[["CVEGEO_MZA", "CVEGEO", "slug", "POBTOT", "distance_m_supermarket_club", "distance_m_fresh_only"]].merge(
    t04[["CVEGEO", "veh_share"]], on="CVEGEO", how="left")
vb = vb[(vb["POBTOT"] > 0) & vb["veh_share"].notna() & vb["distance_m_supermarket_club"].notna()].reset_index(drop=True)
vb["carless"] = 1.0 - vb["veh_share"]
pop_v = vb["POBTOT"].to_numpy(float)
R.p(f"{len(vb):,} inhabited blocks holding {pop_v.sum():,.0f} people carry a vehicle share and a supermarket distance.")
reg("v6.blocks", len(vb), "07_block_access.parquet + 04_tracts.parquet", "veh_share, distance_m_supermarket_club", "blocks in the vehicle table")
reg("v6.population", pop_v.sum(), "07_block_access.parquet + 04_tracts.parquet", "POBTOT", "population in the vehicle table")

cz = vb["carless"].to_numpy(float)
o_ = np.argsort(cz, kind="stable"); cw = np.cumsum(pop_v[o_])
cuts_v = [cz[o_][np.searchsorted(cw, q * cw[-1])] for q in (0.2, 0.4, 0.6, 0.8)]
vb["fifth"] = np.searchsorted(cuts_v, cz, side="left") + 1
far = vb["distance_m_supermarket_club"].to_numpy(float) > MILE_M
maj = cz > 0.5
V6A = []
for k in range(1, 6):
    sel_ = (vb["fifth"] == k).to_numpy()
    pp = pop_v[sel_]
    fr = vb.loc[sel_, "distance_m_fresh_only"].to_numpy(float)
    V6A.append(dict(fifth=k, population=pp.sum(), carless=wmedian(cz[sel_], pp),
                    sup_m=wmedian(vb.loc[sel_, "distance_m_supermarket_club"], pp),
                    far=pp[far[sel_]].sum() / pp.sum(),
                    fresh_m=wmedian(fr, pp),
                    fresh500=pp[np.isfinite(fr) & (fr <= SHORT_M)].sum() / pp[np.isfinite(fr)].sum()))
V6A = pd.DataFrame(V6A)
R.p("Distance by fifth of the share of dwellings without a car, fewest without a car first:")
R.df(V6A)
for _, r_ in V6A.iterrows():
    k = int(r_.fifth)
    for c_ in ("population", "carless", "sup_m", "far", "fresh_m", "fresh500"):
        reg(f"v6.f{k}.{c_}", r_[c_], "07_block_access.parquet + 04_tracts.parquet", "veh_share, distance_m", f"vehicle fifth {k}, {c_}")

def joint_row(sel_, label):
    pp = pop_v[sel_]
    p_far = pp[far[sel_]].sum() / pp.sum(); p_maj = pp[maj[sel_]].sum() / pp.sum()
    p_both = pp[far[sel_] & maj[sel_]].sum() / pp.sum()
    return dict(stratum=label, population=pp.sum(), far=p_far, carless=p_maj, both=p_both, expected=p_far * p_maj,
                ratio=p_both / (p_far * p_maj) if p_far * p_maj > 0 else np.nan)
reg_v = vb["slug"].map(slug_region) if not slug_region.empty else pd.Series("All", index=vb.index)
V6B = [joint_row(np.ones(len(vb), bool), "All 92 areas")]
for rg in [r_ for r_ in ("North", "Centre", "South") if (reg_v == r_).any()]:
    V6B.append(joint_row((reg_v == rg).to_numpy(), rg))
V6B = pd.DataFrame(V6B)
R.p("Population beyond one mile from a supermarket and in a tract where most dwellings have no car:")
R.df(V6B)
for _, r_ in V6B.iterrows():
    tag = r_.stratum.split()[0].lower()
    for c_ in ("far", "carless", "both", "expected"):
        reg(f"v6.{c_}" if tag == "all" else f"v6.{tag}.{c_}", 100 * r_[c_], "07_block_access.parquet + 04_tracts.parquet", "veh_share, distance_m_supermarket_club", f"{r_.stratum}, {c_}, per cent")
    reg(f"v6.ratio" if tag == "all" else f"v6.{tag}.ratio", r_.ratio, "07_block_access.parquet + 04_tracts.parquet", "veh_share, distance_m_supermarket_club", f"{r_.stratum}, observed over expected")

# interaction of deprivation with the carless share, both centred within areas
okc = np.isfinite(carless)
gk = pd.factorize(gcode[okc])[0]
xw = np.full(len(xdep), np.nan); cw_ = np.full(len(xdep), np.nan)
xw[okc] = demean(xdep[okc], w[okc], gk)[:, 0]
cw_[okc] = demean(carless[okc], w[okc], gk)[:, 0]
r_xc = float(np.average(xw[okc] * cw_[okc], weights=w[okc])
             / math.sqrt(np.average(xw[okc] ** 2, weights=w[okc]) * np.average(cw_[okc] ** 2, weights=w[okc])))
R.p(f"Within-area correlation of the index and the carless share, population weighted: {r_xc:.3f}.")
reg("v6.model.corr_within", r_xc, "08_tract_access.parquet", "imu_depriv_z, veh_share", "within-area correlation of index and carless share")
V6C, V6Q = {}, {}
for s in ANCHORS:
    Xc = np.column_stack([xw, cw_, xw * cw_, logden])
    f = fit_on(Y[s], Xc, okc)
    V6C[s] = f
    fq = fit_on(Y[s], np.column_stack([xw, cw_, xw * cw_, logden, xw ** 2]), okc)
    V6Q[s] = fq
    reg(f"v6.quad.inter.{s}", fq["beta"][2], "08_tract_access.parquet", "imu_depriv_z, veh_share", f"interaction with the squared index added, {s}")
    reg(f"v6.quad.inter_se.{s}", fq["se"][2], "08_tract_access.parquet", "imu_depriv_z, veh_share", f"its standard error, {s}")
    reg(f"v6.quad.sq.{s}", fq["beta"][4], "08_tract_access.parquet", "imu_depriv_z", f"coefficient on the squared index, {s}")
    reg(f"v6.quad.sq_se.{s}", fq["se"][4], "08_tract_access.parquet", "imu_depriv_z", f"its standard error, {s}")
    R.p(f"{LABEL[s]}, squared index added: deprivation {fq['beta'][0]:+.4f} ({fq['se'][0]:.4f}), interaction {fq['beta'][2]:+.4f} "
        f"({fq['se'][2]:.4f}), squared index {fq['beta'][4]:+.4f} ({fq['se'][4]:.4f}).")
    for j, nm in enumerate(("dep", "carless", "inter", "den")):
        reg(f"v6.model.{nm}.{s}", f["beta"][j], "08_tract_access.parquet", "imu_depriv_z, veh_share, pop_density_km2", f"interaction model, {nm}, {s}")
        reg(f"v6.model.{nm}_se.{s}", f["se"][j], "08_tract_access.parquet", "imu_depriv_z, veh_share, pop_density_km2", f"interaction model se, {nm}, {s}")
    R.p(f"{LABEL[s]}: deprivation {f['beta'][0]:+.4f} ({f['se'][0]:.4f}), carless share {f['beta'][1]:+.4f} ({f['se'][1]:.4f}), "
        f"interaction {f['beta'][2]:+.4f} ({f['se'][2]:.4f}), log density {f['beta'][3]:+.4f} ({f['se'][3]:.4f}), {f['n']:,} tracts.")
sd_c = float(math.sqrt(np.average(cw_[okc] ** 2, weights=w[okc])))
reg("v6.model.carless_sd", sd_c, "08_tract_access.parquet", "veh_share", "weighted within-area sd of the carless share")
for s in ANCHORS:
    b_, V_ = V6C[s]["beta"], V6C[s]["V"]
    for sign, tag in ((1, "high"), (-1, "low")):
        e_ = b_[0] + sign * sd_c * b_[2]
        se_ = math.sqrt(V_[0, 0] + sd_c ** 2 * V_[2, 2] + 2 * sign * sd_c * V_[0, 2])
        reg(f"v6.model.dep_at_{tag}.{s}", e_, "08_tract_access.parquet", "imu_depriv_z, veh_share", f"deprivation coefficient one within-area sd {tag}, {s}")
        reg(f"v6.model.dep_at_{tag}_se.{s}", se_, "08_tract_access.parquet", "imu_depriv_z, veh_share", f"its standard error, {s}")
        R.p(f"{LABEL[s]}: deprivation coefficient one within-area standard deviation of the carless share {'above' if sign > 0 else 'below'} "
            f"the area mean {e_:+.4f} ({se_:.4f}).")

names5 = {1: "Fewest without a car", 2: "Second fifth", 3: "Third fifth", 4: "Fourth fifth", 5: "Most without a car"}
hd = [["", "Population (millions)", "Median share without a car", r"Supermarkets, median (m)", r"Beyond one mile (\%)",
       r"Fresh, median (m)", r"Fresh within 500 m (\%)"]]
bd = [panel_row(r"Panel A. Residents by fifth of the share of dwellings in their tract without a car", 7)]
for _, r_ in V6A.iterrows():
    bd.append([names5[int(r_.fifth)], num(r_.population / 1e6, 1), num(r_.carless, 3), num(r_.sup_m), pct(r_.far), num(r_.fresh_m), pct(r_.fresh500)])
bd.append(r"\hline")
bd.append(panel_row(r"Panel B. Beyond one mile from a supermarket and in a tract where most dwellings have no car", 7))
bd.append(r" & Population (millions) & Beyond one mile (\%) & Majority without a car (\%) & Both (\%) & Expected if unrelated (\%) & Ratio \\ \hline")
for _, r_ in V6B.iterrows():
    bd.append([tex(r_.stratum), num(r_.population / 1e6, 1), pct(r_.far), pct(r_.carless), pct(r_.both), pct(r_.expected), num(r_.ratio, 2)])
bd.append(r"\hline")
bd.append(panel_row(r"Panel C. Within-area model with the carless share and its interaction with deprivation", 7))
bd.append(r" & Deprivation & Share without a car & Deprivation $\times$ share & Log density & Tracts & \\ \hline")
for s in ANCHORS:
    f = V6C[s]
    bd.append([tex(SHORT[s]), num(f["beta"][0], 4), num(f["beta"][1], 4), num(f["beta"][2], 4), num(f["beta"][3], 4), num(f["n"]), ""])
    bd.append(["", "(" + num(f["se"][0], 4) + ")", "(" + num(f["se"][1], 4) + ")", "(" + num(f["se"][2], 4) + ")", "(" + num(f["se"][3], 4) + ")", "", ""])
notes = (f"Panel A divides the {pop_v.sum() / 1e6:.1f} million residents of inhabited blocks with a vehicle share and a supermarket "
         f"distance into fifths of equal population by the share of dwellings in their tract without a car; distances are population "
         f"weighted medians over the fifth. Panel B counts residents beyond {MILE_M:,.0f} m of a supermarket or club, residents in tracts "
         f"where more than half of dwellings have no car, and residents meeting both conditions, against the product of the two shares. "
         f"Panel C adds to the model of the gradient table the share of dwellings without a car and its product with the standardised "
         f"index, both centred on their population weighted mean within each area (within-area standard deviation of the share "
         f"{sd_c:.3f}), on the {V6C[ANCHORS[0]]['n']:,} tracts of the estimation sample with a vehicle share; errors clustered on "
         f"areas in parentheses.")
write_table(os.path.join(DIRS["tables_main"], "t6_vehicle.tex"),
            "Walking distance to food and vehicle availability",
            "tab:vehicle", "|l|r|r|r|r|r|r|", hd, bd, notes)

## 9. Tables 1 and 2, the definitions and walking distance

Table 1 counts establishments and distinct coordinates in each set from the establishment register, and lists the activity codes it contains. Street sets add the blocks on which street commerce was recorded on at least one frontage, among blocks where the audit recorded it. Table 2 gives population weighted medians of block distance and walking time and the population shares within 500 metres and one mile, over the population whose block reaches the set through the network.

In [ ]:
R.h("Tables 1 and 2")
out["codigo_act"] = out["codigo_act"].astype(str).str.extract(r"(\d{6})", expand=False)
SETMASK, DERIVED = {}, {}
for s in SETS:
    if s in STREET_BASE:
        continue
    if s in out.columns:
        SETMASK[s] = out[s].fillna(False).astype(bool)
    elif s == "supermarket_mini_pharmacy" and {"supermarket_mini", "pharmacy_mini"} <= set(out.columns):
        SETMASK[s] = out["supermarket_mini"].fillna(False).astype(bool) | out["pharmacy_mini"].fillna(False).astype(bool)
        DERIVED[s] = "supermarket_mini or pharmacy_mini"
    elif s == "fresh_perishable" and "fresh_only" in out.columns:
        tb = out["nombre_act"].astype(str).str.contains(r"tortill|panader|nixtamal|\bpan\b", case=False, regex=True)
        SETMASK[s] = out["fresh_only"].fillna(False).astype(bool) & ~tb
        DERIVED[s] = "fresh_only without tortilla, nixtamal and bakery activities"
    else:
        DERIVED[s] = "not available in 05_outlets"
for s, how in DERIVED.items():
    R.p(f"- `{s}`: {how}.")

def code_list(codes):
    codes = sorted(set(c for c in codes if isinstance(c, str)))
    if not codes:
        return "--"
    if len(codes) <= 3:
        return ", ".join(codes)
    grp = pd.Series(codes).str[:4].value_counts().sort_index()
    return ", ".join(grp.index) + f" ({len(codes)} codes)"

street_mask = stc["street_any"].fillna(False).astype(bool) & stc["street_known"].fillna(False).astype(bool)
n_street = int(street_mask.sum())
reg("t1.street_blocks", n_street, "06_street_commerce.parquet", "street_any & street_known", "blocks carrying street commerce")
def n_locations(msk):
    sub = out.loc[msk]
    res = {}
    if {"latitud", "longitud"} <= set(out.columns):
        res["register coordinates, 5 decimals"] = int(sub[["latitud", "longitud"]].round(5).drop_duplicates().shape[0])
        res["register coordinates, as published"] = int(sub[["latitud", "longitud"]].drop_duplicates().shape[0])
    res["projected, 1 m"] = int(sub[["x_m", "y_m"]].round(0).drop_duplicates().shape[0])
    return res
LOC_ALT = []
# Five decimals of a degree are about one metre, the key the outlet notebook uses to count locations.
LOC_KEY = "register coordinates, 5 decimals" if {"latitud", "longitud"} <= set(out.columns) else "projected, 1 m"

if "fresh_counter" in SETMASK and "fresh_only" in SETMASK:
    extra = SETMASK["fresh_counter"] & ~SETMASK["fresh_only"]
    R.p(f"Establishments in `fresh_counter` and not in `fresh_only`: {int(extra.sum()):,}, by activity:")
    R.df(out.loc[extra].groupby(["codigo_act", "nombre_act"]).size().rename("establishments").reset_index())
    if "dairy_craft" in out.columns and int(extra.sum()) and bool(out.loc[extra, "dairy_craft"].fillna(False).all()):
        CONTENTS["fresh_counter"] = "Fresh and staple specialists with craft dairy producers"
        R.p("All of them carry the `dairy_craft` flag; Table 1 describes the set accordingly.")
    else:
        R.p("They do not all carry the `dairy_craft` flag; the Table 1 description of `fresh_counter` should be checked against the list above.")

T1 = []
for s in SETS:
    base = STREET_BASE.get(s, s)
    if base in SETMASK:
        msk = SETMASK[base]
        est = int(msk.sum())
        alts = n_locations(msk)
        loc = alts[LOC_KEY]
        LOC_ALT.append(dict(set=s, establishments=int(msk.sum()), **alts))
        codes = code_list(out.loc[msk, "codigo_act"])
    else:
        est = loc = np.nan; codes = "--"
    T1.append(dict(set=s, establishments=est, locations=loc, street_blocks=n_street if s in STREET_BASE else np.nan, codes=codes))
    if s not in STREET_BASE:
        reg(f"t1.estab.{s}", est, "05_outlets.parquet", base, f"establishments in {s}")
        reg(f"t1.loc.{s}", loc, "05_outlets.parquet", "latitud, longitud rounded to 5 decimals" if LOC_KEY.startswith("register") else "x_m, y_m", f"distinct locations in {s}")
T1 = pd.DataFrame(T1)
R.df(T1)
R.p(f"Distinct locations under three coordinate keys; Table 1 uses {LOC_KEY}:")
R.df(pd.DataFrame(LOC_ALT))
n_drink = int(out["drinking"].fillna(False).astype(bool).sum()) if "drinking" in out.columns else 0
reg("t1.drinking", n_drink, "05_outlets.parquet", "drinking", "bars, cantinas and nightclubs in no set, within the boundaries")
hd = [["Definition", "Branch", "Contents", "Activity codes (SCIAN)", "Establishments", "Locations", "Street blocks"]]
bd = [[tex(SHORT[r_.set]), tex(BRANCH[r_.set]), r"\raggedright " + tex(CONTENTS[r_.set]), tex(r_.codes) if r_.set not in STREET_BASE else tex("as " + SHORT[STREET_BASE[r_.set]]),
       num(r_.establishments), num(r_.locations), num(r_.street_blocks) if np.isfinite(r_.street_blocks) else ""]
      for _, r_ in T1.iterrows()]
notes = ("Establishments are records of the National Statistical Directory of Economic Units located inside the metropolitan "
         "boundaries. Locations count distinct coordinates to five decimals of a degree, about one metre, since a public market enters "
         "the register once per stall at one point. The self-service branch and the traditional branch each add to the set before them; "
         "the two extensions of the fresh and staple set do not contain one another; and the union contains every other set. Street sets "
         "add the blocks on which the census urban environment audit recorded semi-fixed stalls or itinerant vendors on at least one "
         f"frontage, so their added members are blocks rather than establishments. Bars, cantinas and nightclubs ({n_drink:,} records "
         "inside the boundaries) belong to no set. Where a set holds more than three activity codes the four-digit industry groups are "
         "listed, with the number of six-digit codes in parentheses.")
write_table(os.path.join(DIRS["tables_main"], "t1_definitions.tex"), "Destination sets used to measure walking access to food",
            "tab:definitions", "|l|l|p{5.6cm}|l|r|r|r|", hd, bd, notes)

wb = blk["POBTOT"].to_numpy(float)
T2 = []
for s in SETS:
    dist = blk[f"distance_m_{s}"].to_numpy(float); mins = blk[f"walk_min_{s}"].to_numpy(float)
    ok_ = np.isfinite(dist) & (wb > 0)
    T2.append(dict(set=s, median_m=wmedian(dist, wb), median_min=wmedian(mins, wb),
                   within500=wb[ok_ & (dist <= SHORT_M)].sum() / wb[ok_].sum(),
                   within_mile=wb[ok_ & (dist <= MILE_M)].sum() / wb[ok_].sum(),
                   unreached=wb[~np.isfinite(dist) & (wb > 0)].sum() / wb[wb > 0].sum()))
T2 = pd.DataFrame(T2)
speed = np.nanmedian(blk["distance_m_supermarket_club"] / blk["walk_min_supermarket_club"]) * 60 / 1000
R.df(T2)
R.p(f"Walking speed implied by distance and time {speed:.2f} km/h.")
for _, r_ in T2.iterrows():
    reg(f"t2.median_m.{r_.set}", r_.median_m, "07_block_access.parquet", f"distance_m_{r_.set}", "population weighted median distance")
    reg(f"t2.median_min.{r_.set}", r_.median_min, "07_block_access.parquet", f"walk_min_{r_.set}", "population weighted median walking time")
    reg(f"t2.within500.{r_.set}", 100 * r_.within500, "07_block_access.parquet", f"distance_m_{r_.set}", "per cent within 500 m")
    reg(f"t2.within_mile.{r_.set}", 100 * r_.within_mile, "07_block_access.parquet", f"distance_m_{r_.set}", "per cent within one mile")
    reg(f"t2.unreached.{r_.set}", 100 * r_.unreached, "07_block_access.parquet", f"distance_m_{r_.set}", "per cent of population not reached")
hd = [["Definition", "Median distance (m)", "Median time (min)", r"Within 500 m (\%)", r"Within 1,609 m (\%)"]]
bd = [[tex(LABEL[r_.set]), num(r_.median_m), num(r_.median_min, 1), pct(r_.within500), pct(r_.within_mile)] for _, r_ in T2.iterrows()]
notes = (f"Distances are measured over pedestrian street networks from each of the {len(blk):,} inhabited census blocks to the nearest "
         f"destination in the set, and weighted by block population. Walking time assumes a constant speed of {speed:.1f} km/h. Shares are "
         f"of the population whose block reaches the set through the network. The 1,609 m threshold is the one mile criterion of United "
         f"States federal food access measures, and 500 m is about six minutes of walking at the assumed speed.")
write_table(os.path.join(DIRS["tables_main"], "t2_access.tex"), "Population weighted walking distance to the nearest food destination, by definition",
            "tab:access", "|l|r|r|r|r|", hd, bd, notes)

R.h("Distance predicted by destination density alone", 3)
R.p("For destinations scattered at random with intensity proportional to their number of distinct locations, the median distance to the nearest one scales with the inverse square root of that number. The predicted ratio of median distances between the narrowest set and each wider set is therefore the square root of their ratio of locations.")
LOCS = T1.set_index("set")["locations"]
MEDS = T2.set_index("set")["median_m"]
DN = []
for s in ["supermarket_mini", "supermarket_mini_pharmacy", "fresh_perishable", "fresh_only", "fresh_counter", "fresh_abarrotes", "all_food"]:
    lr = LOCS[s] / LOCS["supermarket_club"]
    obs = MEDS["supermarket_club"] / MEDS[s]
    DN.append(dict(set=s, location_ratio=lr, predicted=math.sqrt(lr), observed=obs, observed_over_predicted=obs / math.sqrt(lr)))
    for k_, v in (("location_ratio", lr), ("predicted", math.sqrt(lr)), ("observed", obs), ("ratio", obs / math.sqrt(lr))):
        reg(f"dnull.{k_}.{s}", v, "05_outlets.parquet + 07_block_access.parquet", f"latitud, longitud, distance_m_{s}", f"density null, {k_}, {s} against supermarket_club")
R.df(pd.DataFrame(DN))

R.h("Share of variance in log distance between areas", 3)
ICC = {}
for s in ANCHORS:
    dist = blk[f"distance_m_{s}"].to_numpy(float); ok_ = np.isfinite(dist) & (wb > 0)
    yv, ww = np.log1p(dist[ok_]), wb[ok_]; gg = pd.factorize(blk.loc[ok_, "slug"])[0]
    mu = np.average(yv, weights=ww); ma = np.bincount(gg, ww * yv) / np.bincount(gg, ww)
    ICC[s] = np.sum(ww * (ma[gg] - mu) ** 2) / np.sum(ww * (yv - mu) ** 2)
    reg(f"icc.{s}", ICC[s], "07_block_access.parquet", f"distance_m_{s}", "share of weighted variance in log(1 + distance) between areas")
    R.p(f"- {LABEL[s]}: {100 * ICC[s]:.1f} per cent of the population weighted variance of log(1 + distance) lies between areas.")

## 10. Appendix tables

Table A1 gathers robustness results for the two anchor definitions: the baseline, perishable specialists in place of fresh and staple specialists, the sets without establishments on a coordinate shared by ten or more register records, tracts at least 99 per cent complete, two other measures of deprivation, weighted median regression with standard errors from resampling whole areas, and standard errors that allow correlation between tracts within 1, 2 and 5 km. It also replaces the index by normal scores of its rank and by the social backwardness grade. Its second panel reports the share of residents on a block carrying street commerce by equal population fifths of deprivation, within areas, with and without control for density. Where notebook 13 has been run, Table A1 also carries the gradient on walking distance adjusted for slope, for the walk to the outlet, and its note gives Moran's I of the baseline residuals. Table A2 gives the gradient by official typology and by macro region. The sample at each stage of construction goes to the supplementary files.

In [ ]:
R.h("Appendix tables")
A1 = {}
for s in ANCHORS:
    A1[(s, "baseline")] = (GR.loc[s, "coef"], GR.loc[s, "se"])
A1[("fresh_only", "perishable")] = (GR.loc["fresh_perishable", "coef"], GR.loc["fresh_perishable", "se"])

for s, base_ in ROB_PAIRS.items():
    f = fit_on(YR[s], np.column_stack([xdep, logden]))
    A1[(base_, "noshared")] = (f["beta"][0], f["se"][0])
    reg(f"a1.noshared.{base_}", f["beta"][0], "07_block_access_robustness.parquet", f"distance_m_{s}", "gradient without establishments on shared coordinates")
    reg(f"a1.noshared_se.{base_}", f["se"][0], "07_block_access_robustness.parquet", f"distance_m_{s}", "its standard error")
    reg(f"a1.noshared_tracts.{base_}", int(f["n"]), "07_block_access_robustness.parquet", f"distance_m_{s}", "tracts with a distance")
    R.p(f"Without shared coordinates, {LABEL[base_]}: {f['beta'][0]:+.4f} ({f['se'][0]:.4f}) on {f['n']:,} tracts.")

if "tract_complete" in d.columns:
    comp = d["tract_complete"].to_numpy(float)
    comp_src = "08_tract_access.parquet"
else:
    comp = d[["CVEGEO"]].merge(t04[["CVEGEO", "tract_complete"]], on="CVEGEO", how="left")["tract_complete"].to_numpy(float)
    comp_src = "04_tracts.parquet"
R.p(f"Tract completeness read from `{comp_src}`; {int(np.isnan(comp).sum())} tracts without a value are kept out of the complete subset.")
sel = comp >= COMPLETE_MIN
for s in ANCHORS:
    f = fit(Y[s][sel], np.column_stack([xdep[sel], logden[sel]]), w[sel], gcode[sel])
    A1[(s, "complete")] = (f["beta"][0], f["se"][0])
    reg(f"a1.complete.{s}", f["beta"][0], comp_src, "tract_complete", "gradient on complete tracts")
reg("a1.complete_tracts", int(sel.sum()), comp_src, "tract_complete", f"tracts at least {COMPLETE_MIN:.0%} complete")

tail = xdep > 3
R.p(f"Tracts with the index above 3 standard deviations: {int(tail.sum()):,}, holding {100 * w[tail].sum() / w.sum():.2f} per cent of the population; maximum {xdep.max():.2f}.")
o_ = np.argsort(xdep, kind="stable")
cw = np.cumsum(w[o_])
pr = np.empty(len(xdep)); pr[o_] = (cw - 0.5 * w[o_]) / cw[-1]
xns = stats.norm.ppf(pr)
for s in ANCHORS:
    f = fit(Y[s], np.column_stack([xns, logden]), w, gcode)
    A1[(s, "rank")] = (f["beta"][0], f["se"][0])
    reg(f"a1.rank.{s}", f["beta"][0], "08_tract_access.parquet", "imu_depriv_z", "gradient on normal scores of the population weighted rank")
if "grs_grade_ord" in tr.columns:
    okg = ((tr["population"] > 0) & tr["grs_grade_ord"].notna() & (tr["pop_density_km2"] > 0)
           & tr[med_cols].notna().all(axis=1))
    dg = tr.loc[okg]
    for s in ANCHORS:
        f = fit(np.log1p(dg[f"median_m_{s}"].to_numpy(float)),
                np.column_stack([dg["grs_grade_ord"].to_numpy(float), np.log(dg["pop_density_km2"].to_numpy(float))]),
                dg["population"].to_numpy(float), dg["slug"])
        A1[(s, "grs")] = (f["beta"][0], f["se"][0])
        reg(f"a1.grs.{s}", f["beta"][0], "08_tract_access.parquet", "grs_grade_ord", "gradient per grade of social backwardness")
    GRS_N = (int(okg.sum()), float(dg["population"].sum()))
    reg("a1.grs_tracts", GRS_N[0], "08_tract_access.parquet", "grs_grade_ord", "tracts in the social backwardness fit")
    R.p(f"Social backwardness fit on {GRS_N[0]:,} tracts holding {GRS_N[1]:,.0f} people.")

# weighted median regression: stored estimates if present, otherwise fitted here
MED = {}
med_src = None
for nm, df in (("11_inference.parquet", inf), ("11_spatial.parquet", spa), ("11_contrasts.parquet", con)):
    cb = find_col(df, [r"^(coef|beta|b)_?(median|qr|q50|lad)$", r"^(median|qr|q50|lad)_?(coef|beta|b|est)", r"(median|qr|q50|lad).*(coef|beta|est)"],
                  "median coefficient", nm, exclude=r"(median_m|se|lo|hi|ci)", required=False)
    if cb is None:
        continue
    cse = find_col(df, [r"boot.*se", r"se.*boot", r"se_?(median|qr|q50|lad)", r"(median|qr|q50|lad).*se"], "median se", nm, required=False)
    clo = find_col(df, [r"(median|qr|q50|lad|boot).*(lo|lower|l95|025)", r"(lo|lower)_?(median|qr|boot)"], "median lower", nm, required=False)
    chi = find_col(df, [r"(median|qr|q50|lad|boot).*(hi|upper|u95|975)", r"(hi|upper)_?(median|qr|boot)"], "median upper", nm, required=False)
    scn = set_columns(df)[0]
    for s in ANCHORS:
        r_ = df[df[scn].astype(str) == s]
        if len(r_):
            r_ = r_.iloc[0]
            MED[s] = (r_[cb], r_[cse] if cse else np.nan, r_[clo] if clo else np.nan, r_[chi] if chi else np.nan)
    med_src = (nm, cb, cse)
    break
if not all(s in MED and np.isfinite(MED[s][1]) for s in ANCHORS):
    R.p(f"Median regression is fitted here, with {N_BOOT} resamples of whole areas.")
    cache = os.path.join(DIRS["cache"], "median_regression_bootstrap.csv")
    X2 = np.column_stack([xdep, logden])
    point = {s: lad_fe(Y[s], X2, w, gcode)[0] for s in ANCHORS}
    done = pd.read_csv(cache) if os.path.exists(cache) else pd.DataFrame(columns=["draw", *ANCHORS])
    have = set(done["draw"].astype(int))
    rows_b = done.to_dict("records")
    idx_by = [np.flatnonzero(gcode == a) for a in range(len(AREAS))]
    t_b = time.time()
    for bb in range(N_BOOT):
        if bb in have:
            continue
        rb = np.random.default_rng([SEED, bb])
        pick = rb.integers(0, len(AREAS), len(AREAS))
        ii = np.concatenate([idx_by[a] for a in pick])
        gb = np.concatenate([np.full(len(idx_by[a]), j) for j, a in enumerate(pick)])
        rows_b.append({"draw": bb, **{s: lad_fe(Y[s][ii], X2[ii], w[ii], gb)[0] for s in ANCHORS}})
        if len(rows_b) % 25 == 0:
            pd.DataFrame(rows_b).to_csv(cache, index=False)
            print(f"  {len(rows_b)} of {N_BOOT} resamples, {time.time() - t_b:,.0f} s")
    bdf = pd.DataFrame(rows_b)
    bdf.to_csv(cache, index=False)
    bdf = bdf[bdf["draw"] < N_BOOT]
    boots = {s: bdf[s].to_numpy(float) for s in ANCHORS}
    R.p(f"{len(bdf)} resamples used, {int(sum(np.isnan(boots[s]).sum() for s in ANCHORS))} failed fits dropped.")
    for s in ANCHORS:
        bs = boots[s][np.isfinite(boots[s])]
        MED[s] = (point[s], np.std(bs, ddof=1), np.percentile(bs, 2.5), np.percentile(bs, 97.5))
    med_src = ("computed in notebook 12", "weighted LAD with area indicators", "bootstrap over areas")
for s in ANCHORS:
    A1[(s, "median")] = MED[s][:2]
    reg(f"a1.median.{s}", MED[s][0], med_src[0], med_src[1], "median regression coefficient")
    reg(f"a1.median_se.{s}", MED[s][1], med_src[0], str(med_src[2]), "median regression bootstrap standard error")
dm = MED[ANCHORS[0]][0] - MED[ANCHORS[1]][0]
R.p(f"Median regression: {ANCHORS[0]} {MED[ANCHORS[0]][0]:.4f}, {ANCHORS[1]} {MED[ANCHORS[1]][0]:.4f}, difference {dm:.4f}.")
reg("a1.median_diff", dm, med_src[0], med_src[1], "difference between the two median coefficients")

# spatially corrected errors: stored if present, otherwise computed here
KSE = {}
try:
    scs = set_columns(spa)[0]
    for km in KERNEL_KM:
        ck = find_col(spa, [rf"(?<![0-9]){km}(\.0)?[ _]?km", rf"km[ _]?{km}(?![0-9])", rf"_{km * 1000}(?![0-9])"], f"kernel se {km} km", "11_spatial.parquet",
                      exclude=r"(^t[ _]|tstat|^t$)")
        for s in ANCHORS:
            KSE[(s, km)] = (float(spa.loc[spa[scs].astype(str) == s, ck].iloc[0]), "11_spatial.parquet", ck)
except Exception as exc:
    R.p(f"Stored kernel errors not read ({exc}); computed here.")
    for s in ANCHORS:
        for km in KERNEL_KM:
            KSE[(s, km)] = (float(kernel_se(OWN[s], coords, km * 1000.0)[0]), "computed in notebook 12", f"Bartlett {km} km")
for s in ANCHORS:
    for km in KERNEL_KM:
        v, sf, scol = KSE[(s, km)]
        A1[(s, f"kernel{km}")] = (GR.loc[s, "coef"], v)
        reg(f"a1.kernel_se_{km}km.{s}", v, sf, scol, f"kernel standard error {km} km")

# street commerce by equal population fifths of deprivation
st = stc.copy()
st["known_pop"] = np.where(st["street_known"].fillna(False), st["POBTOT"], 0.0)
st["street_pop"] = np.where(street_mask, st["POBTOT"], 0.0)
agg = st.groupby("CVEGEO").agg(known_pop=("known_pop", "sum"), street_pop=("street_pop", "sum"), pop=("POBTOT", "sum")).reset_index()
sq = tr[["CVEGEO", "slug", "imu_depriv_z", "pop_density_km2"]].merge(agg, on="CVEGEO")
sq = sq[(sq["known_pop"] > 0) & sq["imu_depriv_z"].notna() & (sq["pop_density_km2"] > 0)].reset_index(drop=True)
sq["share"] = sq["street_pop"] / sq["known_pop"]
z = sq["imu_depriv_z"].to_numpy(float)
o_ = np.argsort(z, kind="stable"); cw = np.cumsum(sq["pop"].to_numpy(float)[o_])
cuts = [z[o_][np.searchsorted(cw, q * cw[-1])] for q in (0.2, 0.4, 0.6, 0.8)]
sq["fifth"] = np.searchsorted(cuts, z, side="left") + 1
wk = sq["known_pop"].to_numpy(float)
D = np.column_stack([(sq["fifth"] == k).to_numpy(float) for k in (2, 3, 4, 5)])
ld = np.log(sq["pop_density_km2"].to_numpy(float))
f_fe = fit(sq["share"].to_numpy(float), D, wk, sq["slug"])
f_den = fit(sq["share"].to_numpy(float), np.column_stack([D, ld]), wk, sq["slug"])
f_tr0 = fit(sq["share"].to_numpy(float), sq["fifth"].to_numpy(float), wk, sq["slug"])
f_tr1 = fit(sq["share"].to_numpy(float), np.column_stack([sq["fifth"].to_numpy(float), ld]), wk, sq["slug"])
STF = []
for k in range(1, 6):
    g_ = sq[sq["fifth"] == k]
    row = dict(fifth=k, tracts=len(g_), population=g_["pop"].sum(), share=g_["street_pop"].sum() / g_["known_pop"].sum(),
               coverage=g_["known_pop"].sum() / g_["pop"].sum(), median_density=wmedian(g_["pop_density_km2"], g_["pop"]))
    if k > 1:
        row.update(diff_fe=f_fe["beta"][k - 2], se_fe=f_fe["se"][k - 2], diff_den=f_den["beta"][k - 2], se_den=f_den["se"][k - 2])
    STF.append(row)
STF = pd.DataFrame(STF)
R.p("Street commerce by equal population fifths of the marginalisation index, least deprived first; differences from the first fifth, within areas:")
R.df(STF)
cover = agg.assign(cov=agg["known_pop"] / agg["pop"].where(agg["pop"] > 0)).merge(tr[["CVEGEO", "imu_depriv_z"]], on="CVEGEO")
cover = cover[np.isfinite(cover["cov"]) & cover["imu_depriv_z"].notna()]
r_cov = wcorr(cover["cov"].to_numpy(float), cover["imu_depriv_z"].to_numpy(float), cover["pop"].to_numpy(float))
nat_cov = agg["known_pop"].sum() / agg["pop"].sum()
def joint(f, k=4):
    b_ = f["beta"][:k]; V_ = f["V"][:k, :k]
    c2 = float(b_ @ np.linalg.solve(V_, b_))
    return c2, float(stats.chi2.sf(c2, k))
J_FE, J_DEN = joint(f_fe), joint(f_den)
TREND = dict(joint_fe=J_FE[0], joint_fe_p=J_FE[1], joint_den=J_DEN[0], joint_den_p=J_DEN[1],
             trend_fe=f_tr0["beta"][0], trend_fe_se=f_tr0["se"][0], trend_fe_t=f_tr0["t"][0],
             trend_den=f_tr1["beta"][0], trend_den_se=f_tr1["se"][0], trend_den_t=f_tr1["t"][0],
             density=f_tr1["beta"][1], tracts=len(sq), population=sq["pop"].sum(), coverage=100 * nat_cov, coverage_corr=r_cov)
R.df(pd.DataFrame([TREND]))
src_st = "06_street_commerce.parquet + 08_tract_access.parquet"
for _, r_ in STF.iterrows():
    k = int(r_.fifth)
    reg(f"street.share.f{k}", 100 * r_.share, src_st, "street_any, street_known, POBTOT", f"per cent on a street commerce block, fifth {k}")
    if k > 1:
        reg(f"street.diff_den.f{k}", 100 * r_.diff_den, src_st, "street_any, street_known, imu_depriv_z", f"difference from fifth 1 with density, fifth {k}, points")
for k_, v in TREND.items():
    reg(f"street.{k_}", v, src_st, "street_any, street_known, POBTOT, imu_depriv_z", f"street commerce, {k_}")

# slope adjusted walking time, from notebook 13 where it has been run
SLOPE = load("13_slope_gradient.parquet", required=False)
if SLOPE is not None:
    for v_ in ("to",):
        for s in ANCHORS:
            r_ = SLOPE[(SLOPE["variant"] == v_) & (SLOPE["destination set"] == s)].iloc[0]
            A1[(s, f"slope_{v_}")] = (float(r_["coefficient"]), float(r_["standard error"]))
            reg(f"a1.slope_{v_}.{s}", r_["coefficient"], "13_slope_gradient.parquet", "coefficient", f"gradient on slope adjusted distance, {v_}")
            reg(f"a1.slope_{v_}_se.{s}", r_["standard error"], "13_slope_gradient.parquet", "standard error", "its standard error")
    R.p("Slope adjusted gradients read from `13_slope_gradient.parquet`.")

lab = {"baseline": f"Baseline, errors clustered on {len(AREAS)} areas", "perishable": "Perishable specialists in place of fresh and staple",
       "noshared": "Establishments on shared coordinates removed",
       "complete": f"Tracts at least {COMPLETE_MIN:.0%} complete".replace("%", r"\%"),
       "median": "Weighted median regression, bootstrap over areas",
       "rank": "Index replaced by normal scores of its rank",
       "grs": "Social backwardness grade in place of the index, per grade",
       "slope_to": "Walking distance adjusted for slope"}
for km in KERNEL_KM:
    lab[f"kernel{km}"] = f"Errors allowing correlation within {km} km"
hd = [["", r"\multicolumn{3}{c|}{" + tex(LABEL[ANCHORS[0]]) + "}", r"\multicolumn{3}{c|}{" + tex(LABEL[ANCHORS[1]]) + "}"],
      ["Variant", "Estimate", "Std. error", "95\\% interval", "Estimate", "Std. error", "95\\% interval"]]
bd = [panel_row("Panel A. Coefficient on the marginalisation index", 7)]
SLOPE_ROWS = ["slope_to"] if SLOPE is not None else []
for v in ["baseline", "perishable", "noshared", "complete", "rank", "grs"] + SLOPE_ROWS + ["median"] + [f"kernel{km}" for km in KERNEL_KM]:
    row = [lab[v]]
    for s in ANCHORS:
        if (s, v) in A1:
            e, se = A1[(s, v)]
            if v == "median" and np.isfinite(MED[s][2]):
                lo, hi = MED[s][2], MED[s][3]
            else:
                lo, hi = e - 1.96 * se, e + 1.96 * se
            row += [num(e, 4), num(se, 4), f"{num(lo, 3)} to {num(hi, 3)}"]
        else:
            row += ["", "", ""]
    bd.append(row)
bd.append(r"\hline")
bd.append(panel_row("Panel B. Residents on a block carrying street commerce, by fifth of the marginalisation index", 7))
bd.append(r" & Share (\%) & \multicolumn{2}{c|}{Difference, area effects} & \multicolumn{2}{c|}{Difference, area effects and density} & Tracts \\ \hline")
names5 = {1: "First fifth, least deprived", 2: "Second fifth", 3: "Third fifth", 4: "Fourth fifth", 5: "Last fifth, most deprived"}
for _, r_ in STF.iterrows():
    k = int(r_.fifth)
    if k == 1:
        bd.append([names5[k], pct(r_.share), r"\multicolumn{2}{c|}{reference}", r"\multicolumn{2}{c|}{reference}", num(r_.tracts)])
    else:
        bd.append([names5[k], pct(r_.share), num(100 * r_.diff_fe, 2), num(100 * r_.se_fe, 2), num(100 * r_.diff_den, 2), num(100 * r_.se_den, 2), num(r_.tracts)])
bd.append([r"Joint test that the four differences are zero, $\chi^2_4$ ($p$)", "",
           r"\multicolumn{2}{c|}{" + num(J_FE[0], 1) + " (" + pfmt(J_FE[1]) + ")}",
           r"\multicolumn{2}{c|}{" + num(J_DEN[0], 1) + " (" + pfmt(J_DEN[1]) + ")}", ""])
slope_note = (" The slope row times each edge of the network in the direction of the walk with Tobler's hiking function, scaled to "
              "4.8 km/h on flat ground, from elevations at 15 m resolution, and expresses the time of the walk from each block to the "
              "nearest outlet as the length of a flat walk of equal duration before the tract median is taken." if SLOPE is not None else "")
moran_note = (f" Moran's $I$ of the baseline residuals, with the eight nearest tracts in the same area as neighbours and values "
              f"centred within areas, is {num(MOR[ANCHORS[0]]['moran_base'], 3)} for {tex(LABEL[ANCHORS[0]]).lower()} and "
              f"{num(MOR[ANCHORS[1]]['moran_base'], 3)} for {tex(LABEL[ANCHORS[1]]).lower()}, "
              + (f"and no one of {N_PERM} rearrangements of values within areas reaches either value."
                 if max(MOR[s]['p_base'] for s in ANCHORS) <= 1.0 / (N_PERM + 1) + 1e-12 else
                 f"with permutation probabilities of at most {max(MOR[s]['p_base'] for s in ANCHORS):.3f}."))
grs_note = (f" The social backwardness row uses the {GRS_N[0]:,} tracts with a grade and every distance." if "grs_grade_ord" in tr.columns else "")
n_sh_sup = int((out["supermarket_club"].fillna(False).astype(bool) & (out["point_records"] >= 10)).sum()) if "point_records" in out.columns else 0
n_sh_fr = int((out["fresh_only"].fillna(False).astype(bool) & (out["point_records"] >= 10)).sum()) if "point_records" in out.columns else 0
reg("a1.noshared_removed.supermarket_club", n_sh_sup, "05_outlets.parquet", "point_records", "supermarkets on shared coordinates")
reg("a1.noshared_removed.fresh_only", n_sh_fr, "05_outlets.parquet", "point_records", "fresh sellers on shared coordinates")
notes = (f"Panel A refers to the model of the gradient table. The shared coordinate row removes establishments whose coordinate is shared "
         f"by ten or more register records, usually the centroid of a locality rather than a shop, although some are public markets: "
         f"{n_sh_sup:,} supermarkets and clubs and {n_sh_fr:,} fresh and staple sellers inside the boundaries. Normal scores replace the index by the standard normal quantile of its "
         f"population weighted rank, which removes the influence of its long upper tail.{grs_note}{slope_note} The median regression interval is the "
         f"percentile interval of {N_BOOT} resamples of whole areas. Kernel errors use a Bartlett kernel over tract centroids within each "
         f"area and are reported for comparison only, since the clustered errors are larger and are the ones used for inference.{moran_note} Panel B "
         f"divides {len(sq):,} tracts into fifths of equal population by the index. The share is the population on a block where street "
         f"commerce was recorded, among blocks where the audit recorded it. Differences from the first fifth, in percentage points, come "
         f"from weighted least squares with area fixed effects, weights equal to the population on recorded blocks and errors clustered on "
         f"areas, without and with log population density. The joint test uses the clustered covariance of the four differences. The audit records street commerce for {100 * nat_cov:.1f} per cent of the "
         f"population, and the correlation between tract coverage and the index is {num(r_cov, 3)}.")
write_table(os.path.join(DIRS["tables_appendix"], "a1_robustness.tex"), "Robustness of the deprivation gradient for the two anchor definitions",
            "tab:robustness", "|l|r|r|c|r|r|c|", hd, bd, notes)

# A2 strata
A2 = []
for kind, mp in (("Typology", slug_tipo), ("Macro region", slug_region)):
    if mp.empty:
        continue
    lev = d["slug"].map(mp)
    for val in [v for v in pd.unique(lev.dropna())]:
        sel_ = (lev == val).to_numpy()
        row = dict(kind=kind, stratum=str(val), areas=int(d.loc[sel_, "slug"].nunique()), tracts=int(sel_.sum()), population=w[sel_].sum())
        for s in ANCHORS:
            f = fit(Y[s][sel_], np.column_stack([xdep[sel_], logden[sel_]]), w[sel_], gcode[sel_])
            row[f"coef_{s}"], row[f"se_{s}"] = f["beta"][0], f["se"][0]
            reg(f"a2.coef.{s}.{val}", f["beta"][0], "08_tract_access.parquet + 08_area_access.parquet", "tipo, macro_region", f"gradient in {val}")
        A2.append(row)
A2 = pd.DataFrame(A2)
R.df(A2)
hd = [["Stratum", "Areas", "Tracts", "Population", r"\multicolumn{2}{c|}{" + tex(SHORT[ANCHORS[0]]) + "}", r"\multicolumn{2}{c|}{" + tex(SHORT[ANCHORS[1]]) + "}"],
      ["", "", "", "", "Coef.", "Std. error", "Coef.", "Std. error"]]
bd = []
for kind in A2["kind"].unique():
    bd.append(panel_row(tex(kind), 8))
    for _, r_ in A2[A2.kind == kind].iterrows():
        bd.append([tex({"Metropoli municipal": "Metrópoli municipal"}.get(r_.stratum, r_.stratum)), num(r_.areas), num(r_.tracts), num(r_.population), num(r_[f"coef_{ANCHORS[0]}"], 4), num(r_[f"se_{ANCHORS[0]}"], 4),
                   num(r_[f"coef_{ANCHORS[1]}"], 4), num(r_[f"se_{ANCHORS[1]}"], 4)])
    bd.append(r"\hline")
bd = bd[:-1]
notes = "The model of the gradient table fitted separately within each stratum, with errors clustered on the areas of the stratum."
write_table(os.path.join(DIRS["tables_appendix"], "a2_strata.tex"), "Deprivation gradient by official metropolitan typology and by macro region",
            "tab:strata", "|l|r|r|r|r|r|r|r|", hd, bd, notes)

# A3 attrition
A3 = []
p03 = os.path.join(RESULTS, "03_blocks.parquet")
if os.path.exists(p03):
    A3.append(dict(stage="Blocks intersecting the 92 metropolitan boundaries", unit="blocks", count=nrows_parquet(p03), population=np.nan,
                   reason="Representative point inside a boundary"))
b04 = load("04_blocks.parquet", columns=["CVEGEO_MZA", "POBTOT"], required=False)
if b04 is not None:
    A3.append(dict(stage="Blocks in the urban census", unit="blocks", count=len(b04), population=b04["POBTOT"].sum(),
                   reason="Blocks outside the census universe do not join"))
A3.append(dict(stage="Inhabited blocks", unit="blocks", count=len(blk), population=blk["POBTOT"].sum(), reason="No origin for a block without residents"))
A3.append(dict(stage="Census tracts", unit="tracts", count=len(t04), population=t04["POBTOT"].sum() if "POBTOT" in t04 else np.nan, reason="Tracts holding frame blocks"))
if "block_weightable" in t04.columns:
    wt = t04["block_weightable"].fillna(False).astype(bool)
    A3.append(dict(stage="Tracts with an inhabited block", unit="tracts", count=int(wt.sum()), population=t04.loc[wt, "POBTOT"].sum() if "POBTOT" in t04 else np.nan,
                   reason="Population weighting undefined otherwise"))
has = tr["imu_depriv_z"].notna() & (tr["population"] > 0)
A3.append(dict(stage="Tracts with a marginalisation score", unit="tracts", count=int(has.sum()), population=tr.loc[has, "population"].sum(), reason="Index not published for the tract"))
A3.append(dict(stage="Estimation sample", unit="tracts", count=len(d), population=w.sum(), reason="Index, population, density and every distance present"))
A3 = pd.DataFrame(A3)
R.df(A3)
for i, r_ in A3.iterrows():
    reg(f"a3.count.{i}", r_["count"], "various", r_.stage, f"attrition: {r_.stage}")
    reg(f"a3.population.{i}", r_["population"], "various", r_.stage, f"attrition population: {r_.stage}")
for k, i in (("frame.blocks_boundary", "Blocks intersecting the 92 metropolitan boundaries"), ("frame.blocks_census", "Blocks in the urban census"),
             ("frame.blocks_inhabited", "Inhabited blocks"), ("frame.tracts", "Census tracts")):
    r_ = A3[A3.stage == i]
    if len(r_):
        reg(k, r_["count"].iloc[0], "attrition", i, i)
reg("frame.population", blk["POBTOT"].sum(), "07_block_access.parquet", "POBTOT", "population of inhabited blocks")
A3.to_csv(os.path.join(DIRS["supplement"], "s6_attrition.csv"), index=False)
R.p("Wrote `supplement/s6_attrition.csv`, the sample at each stage of construction.")

## 11. Figures

Figures are written at 300 dpi without titles, since captions carry them. `f2_gradient.png` shows the within-area coefficient by definition with 95 per cent intervals. `f3_maps.png` places each area at its population weighted tract centroid over state boundaries, since metropolitan polygons are too small to read at national scale; the boundaries come from the national geostatistical framework in `Inputs/MGN`, or from Natural Earth when no state layer is found there. `f4_scatter.png` plots each area's rank under one anchor definition against the other, labelling the six largest moves in each direction with leader lines. The four workflow images show one 1.6 by 1.2 km window of a large city: blocks shaded by population, outlets by type with street commerce blocks highlighted, the pedestrian network, and blocks coloured by walking minutes to the nearest supermarket or club. The window is chosen inside the area by a rule stated in the code rather than by hand.

In [ ]:
R.h("Figures")
def save(fig, name):
    p = os.path.join(DIRS["figures"], name)
    fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    R.p(f"Wrote `figures/{name}`.")

# Figure 2
fig, ax = plt.subplots(figsize=(COL_W, 3.0))
G2 = GR.loc[SETS[::-1]]
yy = np.arange(len(G2))
lo, hi = G2["coef"] - 1.96 * G2["se"], G2["coef"] + 1.96 * G2["se"]
for yi, (s, r_) in zip(yy, G2.iterrows()):
    col = "#8c2d04" if r_.coef > 0 else "#2b6a99"
    cross = lo[s] <= 0 <= hi[s]
    ax.plot([lo[s], hi[s]], [yi, yi], color=col, lw=1.1)
    ax.plot(r_.coef, yi, "o", ms=3.5, color=col, mfc="white" if cross else col)
ax.axvline(0, color="#777777", lw=0.6)
ax.set_yticks(yy); ax.set_yticklabels([SHORT[s] for s in G2.index])
ax.set_xlabel("Coefficient on marginalisation index (log distance)")
save(fig, "f2_gradient.png")

# area centroids and ranks
cen = tr.dropna(subset=["x_m", "y_m"]).groupby("slug").apply(
    lambda g: pd.Series({"x": np.average(g["x_m"], weights=g["population"].clip(lower=1e-9)),
                         "y": np.average(g["y_m"], weights=g["population"].clip(lower=1e-9)),
                         "pop": g["population"].sum()}), include_groups=False)
cen["rank_a"] = RANK[ANCHORS[0]].reindex(cen.index)
cen["rank_b"] = RANK[ANCHORS[1]].reindex(cen.index)
cen["region"] = slug_region.reindex(cen.index) if not slug_region.empty else "All"

# Figure 3
def geom_lines(geom):
    out = []
    for g in getattr(geom, "geoms", [geom]):
        if g.geom_type == "Polygon":
            out.append(np.asarray(g.exterior.coords))
            out += [np.asarray(r.coords) for r in g.interiors]
        elif g.geom_type in ("LineString", "LinearRing"):
            out.append(np.asarray(g.coords))
        elif hasattr(g, "geoms"):
            out += geom_lines(g)
    return out

def load_outline():
    """State polygons in EPSG:6372, from the national geostatistical framework in Inputs/MGN, else Natural Earth."""
    try:
        import geopandas as gpd
    except Exception as exc:
        return None, f"geopandas not available ({exc})"
    cache = os.path.join(DIRS["cache"], "mexico_states_6372.gpkg")
    if os.path.exists(cache):
        return gpd.read_file(cache), "cached state boundaries"
    pat = re.compile(r"^(\d{2})?ent(idad(es)?)?(_[^.]*)?\.(shp|gpkg|geojson|json)$", re.I)
    found = []
    for dp, dn, fns in os.walk(os.path.join(BASE, "Inputs", "MGN")):
        for fn in fns:
            full = os.path.join(dp, fn)
            if pat.match(fn):
                found.append((full, None))
            elif fn.lower().endswith(".zip"):
                try:
                    with zipfile.ZipFile(full) as z:
                        found += [(f"zip://{full}!{nm}", None) for nm in z.namelist() if pat.match(os.path.basename(nm))]
                except Exception:
                    pass
            elif fn.lower().endswith(".gpkg"):
                try:
                    import pyogrio
                    found += [(full, lyr) for lyr in pyogrio.list_layers(full)[:, 0] if re.match(r"^(\d{2})?ent(idad(es)?)?$", str(lyr), re.I)]
                except Exception:
                    pass
    national = [f for f in found if re.search(r"(^|[/!])00ent", f[0], re.I) or (f[1] or "").lower().startswith("00")]
    sources = []
    for cand in ([national[:1]] if national else []) + ([found] if found else []):
        try:
            parts = [gpd.read_file(p, layer=l) if l else gpd.read_file(p) for p, l in cand]
            g = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs).to_crs(6372)
            km2 = g.geometry.area.sum() / 1e6
            if 1.5e6 < km2 < 2.5e6:
                sources.append((g, "INEGI national geostatistical framework, " + ", ".join(os.path.basename(p) for p, _ in cand[:3])))
                break
        except Exception:
            continue
    if not sources:
        for url, col, val in (("https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_1_states_provinces.zip", "adm0_a3", "MEX"),
                              ("https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip", "ADM0_A3", "MEX")):
            try:
                g = gpd.read_file(url)
                g = g[g[col] == val].to_crs(6372)
                if len(g):
                    sources.append((g, "Natural Earth, " + os.path.basename(url)))
                    break
            except Exception:
                continue
    if not sources:
        return None, "no boundary source found"
    g, how = sources[0]
    g = gpd.GeoDataFrame(geometry=g.geometry.buffer(0).simplify(500), crs=g.crs)
    try:
        g.to_file(cache, driver="GPKG")
    except Exception:
        pass
    return g, how

states, outline_src = load_outline()
R.p(f"Map outline: {outline_src}.")
STATE_LINES, COUNTRY_LINES = [], []
if states is not None:
    for geom in states.geometry:
        STATE_LINES += geom_lines(geom)
    union = states.union_all() if hasattr(states, "union_all") else states.unary_union
    COUNTRY_LINES = geom_lines(union)

nA = int(cen["rank_a"].max())
pmax = cen["pop"].max()
def dot_size(p):
    return 5 + 75 * np.sqrt(np.asarray(p, float) / pmax)
fig, axs = plt.subplots(1, 2, figsize=(2 * COL_W, 2.75), gridspec_kw={"wspace": 0.03})
o = cen.sort_values("pop", ascending=False)
for ax, col, tag in zip(axs, ("rank_a", "rank_b"), ("(a)", "(b)")):
    for arr in STATE_LINES:
        ax.plot(arr[:, 0], arr[:, 1], color="#cfcfcf", lw=0.25, zorder=1)
    for arr in COUNTRY_LINES:
        ax.plot(arr[:, 0], arr[:, 1], color="#5a5a5a", lw=0.5, zorder=2)
    sc = ax.scatter(o["x"], o["y"], c=o[col], s=dot_size(o["pop"]), cmap="viridis", vmin=1, vmax=nA,
                    edgecolor="#222222", linewidth=0.3, zorder=3)
    if COUNTRY_LINES:
        allc = np.vstack(COUNTRY_LINES)
        x0, y0 = allc.min(axis=0); x1, y1 = allc.max(axis=0)
    else:
        x0, y0 = o[["x", "y"]].min(); x1, y1 = o[["x", "y"]].max()
    mx, my = 0.02 * (x1 - x0), 0.02 * (y1 - y0)
    ax.set_xlim(x0 - mx, x1 + mx); ax.set_ylim(y0 - my, y1 + my)
    ax.set_aspect("equal"); ax.axis("off")
    ax.text(0.0, 1.01, tag + " " + SHORT[ANCHORS[0] if col == "rank_a" else ANCHORS[1]], transform=ax.transAxes,
            fontsize=7, va="bottom", ha="left")
ref_pop = [p for p in (0.5e6, 2e6, 10e6) if p <= pmax] or [pmax]
handles = [axs[0].scatter([], [], s=dot_size(p), facecolor="white", edgecolor="#222222", linewidth=0.3) for p in ref_pop]
axs[0].legend(handles, [f"{p / 1e6:g} million" for p in ref_pop], title="Population", loc="lower left", frameon=False,
              fontsize=6, title_fontsize=6, labelspacing=1.1, borderpad=0.3, handletextpad=1.0)
cb = fig.colorbar(sc, ax=axs, orientation="horizontal", fraction=0.05, pad=0.02, aspect=50, shrink=0.6)
cb.set_label("Rank of the area under each definition, 1 = longest median walk"); cb.set_ticks([1, 23, 46, 69, nA])
cb.outline.set_linewidth(0.4)
save(fig, "f3_maps.png")

# Figure 4
def place_labels(ax, xy, labels, others, fontsize=5.6, radii=(7, 13, 20, 28, 37, 47, 58, 70, 84, 100), n_ang=32):
    """Greedy placement with leader lines. Each label takes the cheapest position on rings of increasing radius around
    its point: inside the axes, clear of labels already placed and of the other labelled points, covering few points,
    and with a leader line that crosses no other leader line."""
    fig_ = ax.figure
    fig_.canvas.draw()
    rend = fig_.canvas.get_renderer()
    axbb = ax.get_window_extent(rend)
    P = ax.transData.transform(np.asarray(others, float))
    M = ax.transData.transform(np.asarray(xy, float))
    k = fig_.dpi / 72
    pad = 1.5 * k
    def cross(p, q, r, s_):
        def o_(a, b, c):
            return np.sign((b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0]))
        return o_(p, q, r) != o_(p, q, s_) and o_(r, s_, p) != o_(r, s_, q)
    def near(p, q, c, tol):
        d = q - p
        t = np.clip(np.dot(c - p, d) / max(np.dot(d, d), 1e-9), 0, 1)
        return np.hypot(*(p + t * d - c)) < tol
    def overlaps(b, c):
        return not (b.x1 < c.x0 or b.x0 > c.x1 or b.y1 < c.y0 or b.y0 > c.y1)
    def seg_hits(p, q, b):
        tt = np.linspace(0.08, 0.92, 25)[:, None]
        pts = p + tt * (q - p)
        return bool(np.any((pts[:, 0] > b.x0) & (pts[:, 0] < b.x1) & (pts[:, 1] > b.y0) & (pts[:, 1] < b.y1)))
    placed, segs = [], []
    order = np.argsort(-np.hypot(*(M - M.mean(axis=0)).T))
    angs = np.linspace(0, 2 * np.pi, n_ang, endpoint=False)
    for i in order:
        (x, y), lab, p0 = xy[i], labels[i], M[i]
        best = None
        for rad in radii:
            for a in angs:
                dx, dy = rad * np.cos(a), rad * np.sin(a)
                ha = "left" if dx > 1 else ("right" if dx < -1 else "center")
                t = ax.annotate(lab, (x, y), xytext=(dx, dy), textcoords="offset points", ha=ha, va="center", fontsize=fontsize)
                bb = t.get_window_extent(rend)
                t.remove()
                bb = matplotlib.transforms.Bbox.from_extents(bb.x0 - pad, bb.y0 - pad, bb.x1 + pad, bb.y1 + pad)
                if bb.x0 < axbb.x0 or bb.x1 > axbb.x1 or bb.y0 < axbb.y0 or bb.y1 > axbb.y1:
                    continue
                if any(overlaps(bb, c) for c in placed):
                    continue
                if any(bb.x0 < m[0] < bb.x1 and bb.y0 < m[1] < bb.y1 for j, m in enumerate(M) if j != i):
                    continue
                inside = int(np.sum((P[:, 0] > bb.x0) & (P[:, 0] < bb.x1) & (P[:, 1] > bb.y0) & (P[:, 1] < bb.y1)))
                q = p0 + np.array([dx, dy]) * k
                ncross = sum(cross(p0, q, a_, b_) for a_, b_ in segs)
                nhit = sum(seg_hits(a_, b_, bb) for a_, b_ in segs) + sum(seg_hits(p0, q, c) for c in placed)
                nnear = sum(near(p0, q, M[j], 4 * k) for j in range(len(M)) if j != i)
                cost = inside * 40 + ncross * 80 + nnear * 60 + nhit * 150 + rad
                if best is None or cost < best[0]:
                    best = (cost, dx, dy, ha, bb)
            if best is not None and best[0] <= rad:
                break
        if best is None:
            R.p(f"No clear position for the label {lab}; it is left out of the figure.")
            continue
        _, dx, dy, ha, bb = best
        placed.append(bb)
        segs.append((p0, p0 + np.array([dx, dy]) * k))
        ax.annotate(lab, (x, y), xytext=(dx, dy), textcoords="offset points", ha=ha, va="center", fontsize=fontsize,
                    arrowprops=dict(arrowstyle="-", lw=0.4, color="#666666", shrinkA=0.5, shrinkB=2.2))

fig, ax = plt.subplots(figsize=(COL_W, 3.5))
pal = ["#1b9e77", "#d95f02", "#7570b3", "#666666"]
for i, (rg, gq) in enumerate(cen.groupby("region")):
    ax.scatter(gq["rank_a"], gq["rank_b"], s=10, color=pal[i % 4], label=str(rg), edgecolor="none", alpha=0.9, zorder=3)
ax.plot([1, nA], [1, nA], color="#999999", lw=0.6, ls="--", zorder=1)
cen["move"] = cen["rank_b"] - cen["rank_a"]
movers = pd.concat([cen.nlargest(6, "move", keep="all"), cen.nsmallest(6, "move", keep="all")])
ax.set_xlim(0, nA + 2); ax.set_ylim(0, nA + 2)
ax.set_xlabel(f"Rank under {SHORT[ANCHORS[0]].lower()}")
ax.set_ylabel(f"Rank under {SHORT[ANCHORS[1]].lower()}")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.1), ncol=4, handletextpad=0.1, columnspacing=0.8)
place_labels(ax, movers[["rank_a", "rank_b"]].to_numpy(float).tolist(), [AREA_NAME.get(s, pretty(s)) for s in movers.index],
             cen[["rank_a", "rank_b"]].dropna().to_numpy(float))
save(fig, "f4_scatter.png")
R.p("Areas labelled in Figure 4, with ranks under the two anchor definitions:")
R.df(movers.assign(area=[AREA_NAME.get(s, pretty(s)) for s in movers.index])[["area", "region", "rank_a", "rank_b", "move"]].reset_index(drop=True))

In [ ]:
R.h("Workflow images", 3)
WF_SLUG = "monterrey"
WF_W, WF_H = 1600.0, 1200.0     # metres, a 4:3 window
WF_CELL = 200.0                 # grid step for candidate windows
WF_SET = "supermarket_club"      # the set the distance panel colours by

import geopandas as gpd, pyogrio, glob as glob_
from shapely.geometry import box as shp_box

bm = blk[blk["slug"] == WF_SLUG].merge(stc[["CVEGEO_MZA", "street_any", "street_known"]], on="CVEGEO_MZA", how="left")
bm["street"] = bm["street_any"].fillna(False).astype(bool) & bm["street_known"].fillna(False).astype(bool)
om = out[out["slug"] == WF_SLUG]
x0, y0 = bm["x_m"].min(), bm["y_m"].min()
nx = int(np.ceil((bm["x_m"].max() - x0) / WF_CELL)) + 1; ny = int(np.ceil((bm["y_m"].max() - y0) / WF_CELL)) + 1
def grid_count(xs, ys, wts=None):
    ix = ((np.asarray(xs) - x0) // WF_CELL).astype(int); iy = ((np.asarray(ys) - y0) // WF_CELL).astype(int)
    keep = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny)
    H = np.zeros((nx, ny))
    np.add.at(H, (ix[keep], iy[keep]), 1.0 if wts is None else np.asarray(wts)[keep])
    return H
def window_sum(H, kx, ky):
    C = np.pad(H, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
    return C[kx:, ky:] - C[:-kx, ky:] - C[kx:, :-ky] + C[:-kx, :-ky]
kx, ky = int(WF_W / WF_CELL), int(WF_H / WF_CELL)
H_blk = window_sum(grid_count(bm["x_m"], bm["y_m"]), kx, ky)
H_sup = window_sum(grid_count(om.loc[om[WF_SET].astype(bool), "x_m"], om.loc[om[WF_SET].astype(bool), "y_m"]), kx, ky)
H_fr = window_sum(grid_count(om.loc[om["fresh_only"].astype(bool), "x_m"], om.loc[om["fresh_only"].astype(bool), "y_m"]), kx, ky)
H_st = window_sum(grid_count(bm.loc[bm["street"], "x_m"], bm.loc[bm["street"], "y_m"]), kx, ky)
# The window holds at least one supermarket, fifteen fresh sellers and ten street commerce blocks, and among those the most
# blocks, so that the four panels each have something to show and the choice does not depend on anyone's judgement.
cand = (H_sup >= 1) & (H_fr >= 15) & (H_st >= 10)
if not cand.any():
    cand = H_sup >= 1
ix, iy = np.unravel_index(np.argmax(np.where(cand, H_blk, -1)), H_blk.shape)
wx0, wy0 = x0 + ix * WF_CELL, y0 + iy * WF_CELL
wx1, wy1 = wx0 + WF_W, wy0 + WF_H
WIN = shp_box(wx0, wy0, wx1, wy1)
R.p(f"Window in {AREA_NAME.get(WF_SLUG, WF_SLUG)}: x {wx0:,.0f} to {wx1:,.0f}, y {wy0:,.0f} to {wy1:,.0f} (EPSG:6372); "
    f"{int(H_blk[ix, iy])} blocks, {int(H_sup[ix, iy])} supermarkets, {int(H_fr[ix, iy])} fresh sellers, {int(H_st[ix, iy])} street commerce blocks.")

def locate_block_layer():
    """The archive member of the national geostatistical framework holding block polygons keyed by a 16 character key."""
    for arch in sorted(glob_.glob(os.path.join(BASE, "Inputs", "MGN", "**", "*.zip"), recursive=True)):
        with zipfile.ZipFile(arch) as zf:
            shps = [n for n in zf.namelist() if n.lower().endswith(".shp")]
        for shp in shps:
            try:
                info = pyogrio.read_info(f"/vsizip/{arch}/{shp}")
            except Exception:
                continue
            if "CVEGEO" not in list(info["fields"]) or "Polygon" not in str(info["geometry_type"]):
                continue
            head_ = pyogrio.read_dataframe(f"/vsizip/{arch}/{shp}", columns=["CVEGEO"], max_features=5, read_geometry=False)
            if head_["CVEGEO"].astype(str).str.len().eq(16).all():
                return arch, shp
    raise RuntimeError("no block layer found under Inputs/MGN")
LOCAL_MGN = "/content/mgn_blocks" if IN_COLAB else os.path.join(DIRS["cache"], "mgn_blocks")
os.makedirs(LOCAL_MGN, exist_ok=True)
shp_local = glob_.glob(os.path.join(LOCAL_MGN, "*.shp"))
if not shp_local:
    arch, shp = locate_block_layer()
    stem = os.path.splitext(os.path.basename(shp))[0]
    with zipfile.ZipFile(arch) as zf:
        for n in zf.namelist():
            if os.path.splitext(os.path.basename(n))[0] == stem:
                with zf.open(n) as fh, open(os.path.join(LOCAL_MGN, os.path.basename(n)), "wb") as fo:
                    shutil.copyfileobj(fh, fo)
    shp_local = glob_.glob(os.path.join(LOCAL_MGN, "*.shp"))
BLOCK_PATH = shp_local[0]
BLOCK_CRS = pyogrio.read_info(BLOCK_PATH)["crs"]
bbox_b = tuple(gpd.GeoSeries([WIN.buffer(50)], crs="EPSG:6372").to_crs(BLOCK_CRS).total_bounds)
POLY = pyogrio.read_dataframe(BLOCK_PATH, columns=["CVEGEO"], bbox=bbox_b).to_crs(6372)
POLY = POLY.rename(columns={"CVEGEO": "CVEGEO_MZA"}).merge(
    blk.loc[blk["slug"] == WF_SLUG, ["CVEGEO_MZA", "POBTOT", f"walk_min_{WF_SET}"]], on="CVEGEO_MZA", how="left")
POLY = POLY.merge(bm[["CVEGEO_MZA", "street"]], on="CVEGEO_MZA", how="left")
POLY = gpd.clip(POLY, WIN)
R.p(f"{len(POLY)} block polygons in the window, {int(POLY['POBTOT'].notna().sum())} of them inhabited and measured.")

netfile = [f for f in glob_.glob(os.path.join(BASE, "Networks", "pedestrian_networks", "*.gpkg"))
           if WF_SLUG in re.sub(r"[^a-z0-9]", "", os.path.basename(f).lower().replace("metro", "").replace("pedestrian", ""))]
assert netfile, f"no network file for {WF_SLUG}"
lyrs = [n for n, k in pyogrio.list_layers(netfile[0]) if "Line" in str(k)]
lyr = "edges" if "edges" in lyrs else lyrs[0]
NET_CRS = pyogrio.read_info(netfile[0], layer=lyr)["crs"]
bbox_n = tuple(gpd.GeoSeries([WIN.buffer(50)], crs="EPSG:6372").to_crs(NET_CRS).total_bounds)
NETW = pyogrio.read_dataframe(netfile[0], layer=lyr, columns=[], bbox=bbox_n).to_crs(6372)
NETW = gpd.clip(NETW, WIN)
R.p(f"{len(NETW)} network ways in the window, {NETW.length.sum() / 1000:.1f} km.")

ow = om[(om["x_m"] >= wx0) & (om["x_m"] <= wx1) & (om["y_m"] >= wy0) & (om["y_m"] <= wy1)].copy()
ow["kind"] = np.select([ow["supermarket_club"].astype(bool), ow["fresh_only"].astype(bool), ow["abarrotes"].astype(bool) if "abarrotes" in ow.columns else np.zeros(len(ow), bool)],
                       ["supermarket", "fresh", "grocery"], default="other")
ow = ow[ow["all_food"].astype(bool)]
R.p("Outlets in the window by kind: " + ", ".join(f"{k} {v}" for k, v in ow["kind"].value_counts().items()) + ".")

def wf_axes():
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.set_position([0, 0, 1, 1]); ax.set_xlim(wx0, wx1); ax.set_ylim(wy0, wy1); ax.set_aspect("equal"); ax.axis("off")
    return fig, ax
def wf_save(fig, name):
    p = os.path.join(DIRS["figures"], name)
    fig.savefig(p, dpi=300, facecolor="white", pad_inches=0)
    plt.close(fig)
    R.p(f"Wrote `figures/{name}`.")

fig, ax = wf_axes()
POLY.plot(ax=ax, column="POBTOT", cmap="YlOrBr", edgecolor="white", linewidth=0.25, missing_kwds={"color": "#e6e6e6"},
          vmin=0, vmax=float(np.nanpercentile(POLY["POBTOT"], 98)))
wf_save(fig, "img_blocks.png")

fig, ax = wf_axes()
POLY.plot(ax=ax, color="#f0f0f0", edgecolor="white", linewidth=0.25)
POLY[POLY["street"].fillna(False)].plot(ax=ax, color="#fdd49e", edgecolor="white", linewidth=0.25)
for kind, col, sz, z in (("other", "#bdbdbd", 4, 3), ("grocery", "#7f7f7f", 5, 4), ("fresh", "#2b6a99", 9, 5), ("supermarket", "#8c2d04", 40, 6)):
    sub = ow[ow["kind"] == kind]
    ax.scatter(sub["x_m"], sub["y_m"], s=sz, color=col, edgecolor="white" if kind == "supermarket" else "none", linewidth=0.4, zorder=z)
wf_save(fig, "img_outlets.png")

fig, ax = wf_axes()
POLY.plot(ax=ax, color="#f7f7f7", edgecolor="none")
NETW.plot(ax=ax, color="#333333", linewidth=0.6)
wf_save(fig, "img_network.png")

fig, ax = wf_axes()
POLY.plot(ax=ax, column=f"walk_min_{WF_SET}", cmap="viridis_r", edgecolor="white", linewidth=0.25, missing_kwds={"color": "#e6e6e6"})
sup = ow[ow["kind"] == "supermarket"]
ax.scatter(sup["x_m"], sup["y_m"], s=40, color="#8c2d04", edgecolor="white", linewidth=0.4, zorder=6)
wf_save(fig, "img_distance.png")
reg("wf.blocks", len(POLY), "Inputs/MGN", "block polygons", "blocks in the workflow window")
reg("wf.minutes_min", float(np.nanmin(POLY[f"walk_min_{WF_SET}"])), "07_block_access.parquet", f"walk_min_{WF_SET}", "shortest walk in the window")
reg("wf.minutes_max", float(np.nanmax(POLY[f"walk_min_{WF_SET}"])), "07_block_access.parquet", f"walk_min_{WF_SET}", "longest walk in the window")

## 12. Supplementary files

`s1_area_level.csv` holds every area level measure with ranks under every definition. `s2_per_area_ranks.csv` holds the within-area rank correlation for each area and pair, with its partial counterpart. `s3_coverage.csv` holds network attachment, unreached population and street commerce coverage by area. `s4_value_registry.csv` holds every number placed in a table, with its source file and column.

In [ ]:
R.h("Supplementary files")
s1 = area.copy()
s1.insert(1, "area", s1["slug"].map(AREA_NAME))
for s in SETS:
    s1[f"rank_{s}"] = s1["slug"].map(RANK[s])
s1.to_csv(os.path.join(DIRS["supplement"], "s1_area_level.csv"), index=False)

s2 = pd.DataFrame(PER_AREA)
s2.to_csv(os.path.join(DIRS["supplement"], "s2_per_area_ranks.csv"), index=False)
rr = s2[(s2.set_a == ANCHORS[0]) & (s2.set_b == ANCHORS[1])].sort_values("rho")
R.p("Lowest and highest within-area correlations for the anchor pair:")
R.df(pd.concat([rr.head(5), rr.tail(5)])[["area", "tracts", "rho", "rho_partial"]])

unr = blk.assign(**{f"unreached_{s}": blk[f"distance_m_{s}"].isna() for s in SETS})
gq = unr.groupby("slug")
s3 = pd.DataFrame({"blocks_measured": gq.size(), "population_measured": gq["POBTOT"].sum()})
for s in SETS:
    s3[f"unreached_pop_share_{s}"] = unr.loc[unr[f"unreached_{s}"]].groupby("slug")["POBTOT"].sum().reindex(s3.index).fillna(0) / s3["population_measured"]
cvr = stc.groupby("slug").apply(lambda g: pd.Series({
    "street_known_pop_share": g.loc[g["street_known"].fillna(False), "POBTOT"].sum() / max(g["POBTOT"].sum(), 1),
    "street_pop_share_known": g.loc[g["street_known"].fillna(False) & g["street_any"].fillna(False), "POBTOT"].sum()
                              / max(g.loc[g["street_known"].fillna(False), "POBTOT"].sum(), 1)}), include_groups=False)
s3 = s3.join(cvr).reset_index()
s3 = att.merge(s3, on="slug", how="outer") if "slug" in att.columns else s3
s3.insert(1, "area", s3["slug"].map(AREA_NAME))
s3.to_csv(os.path.join(DIRS["supplement"], "s3_coverage.csv"), index=False)

s4 = pd.DataFrame(REG).drop_duplicates("key", keep="last")
s4.to_csv(os.path.join(DIRS["supplement"], "s4_value_registry.csv"), index=False)
R.p(f"{len(s4):,} numbers registered with their source file and column.")

## 13. Column map, report and download

The column map records which column of each results file supplied each quantity. The report, the four supplementary files and the table fragments are then packed for download.

In [ ]:
if "R_TAIL" in globals():
    R.lines = R.lines[:R_TAIL]
else:
    R_TAIL = len(R.lines)
R.h("Column map")
R.df(pd.DataFrame([dict(quantity=k, column=v) for k, v in COLMAP.items()]))
R.h("Table fragments", 3)
for sub in ("tables_main", "tables_appendix"):
    for fn in sorted(f for f in os.listdir(DIRS[sub]) if f.endswith(".tex")):
        R.p(f"#### {sub}/{fn}")
        R.p("```latex")
        R.p(open(os.path.join(DIRS[sub], fn), encoding="utf-8").read().rstrip())
        R.p("```")
R.save()
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for sub in ("tables_main", "tables_appendix", "supplement", "figures"):
        for fn in sorted(os.listdir(DIRS[sub])):
            zf.write(os.path.join(DIRS[sub], fn), arcname=f"{sub}/{fn}")
    zf.write(REPORT_PATH, arcname=os.path.basename(REPORT_PATH))
shutil.copy(REPORT_PATH, os.path.join(PAPER, os.path.basename(REPORT_PATH))) if os.path.dirname(REPORT_PATH) != PAPER else None
print(f"Report: {REPORT_PATH}\nOutputs: {ZIP_PATH}")
if IN_COLAB:
    colab_files.download(REPORT_PATH)
    colab_files.download(ZIP_PATH)